# Did the CN replies stop the disinformation from spreading?

**Data used:** the experiment itself (who got a CN reply + Views/Likes/Shares)

**Short answer:** No — the opposite. CN'd tweets got MORE views, not fewer. Likes and shares barely moved.

> ⚠️ **The final ML section (4.7, cells ~130–134) needs `xgboost`**, which may not be installed: `pip install xgboost`. Everything before it runs without it.



# 0. Imports

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then take the field-experiment folder inside it.
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
BASE_DIR = _root / "field-experiment"
if not (BASE_DIR / "data").is_dir():
    raise RuntimeError(
        "Could not locate the field-experiment folder from " + str(Path.cwd()) +
        ". Run this notebook from inside the cloned repository."
    )
DATA_DIR = BASE_DIR / 'data'
OUT_DIR  = BASE_DIR / 'outputs' / '01_main_effect'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data: {DATA_DIR}')
print(f'Out:  {OUT_DIR}')


# 1. Preprocessing

## Load data

In [ ]:
control = pd.read_excel(DATA_DIR / 'Control_Group.xlsx')
treatment = pd.read_excel(DATA_DIR / 'Treatment_Group.xlsx')
monitoring = pd.read_excel(DATA_DIR / 'Tweet Monitoring.xlsx')
users = pd.read_excel(DATA_DIR / 'Existing_Users.xlsx')

print(f"Loaded: Control={len(control)}, Treatment={len(treatment)}, Monitoring={len(monitoring)}")

## Fix date format

In [ ]:
def parse_mixed_date(date_val):
    """
    Handles these formats (experiment ran Dec 2025 / Jan 2026):
    - 2025-12-08 or 2025-08-12 (YYYY-MM-DD or YYYY-DD-MM)
    - 10/12/2025 or 12/10/2025 (DD/MM/YYYY or MM/DD/YYYY)

    Logic: December must be 2025, January must be 2026
    """
    if pd.isna(date_val):
        return pd.NaT

    date_str = str(date_val).strip()

    def is_valid(year, month):
        return (year == 2025 and month == 12) or (year == 2026 and month == 1)

    if '-' in date_str and date_str[:4].isdigit():
        parts = date_str.split('-')
        year = int(parts[0])
        num1 = int(parts[1])
        num2 = int(parts[2].split()[0])

        if is_valid(year, num1):
            month, day = num1, num2
        elif is_valid(year, num2):
            month, day = num2, num1
        else:
            month, day = num1, num2

        return pd.Timestamp(year=year, month=month, day=day)

    elif '/' in date_str:
        parts = date_str.split('/')
        num1 = int(parts[0])
        num2 = int(parts[1])
        year = int(parts[2].split()[0])

        if is_valid(year, num2):
            day, month = num1, num2
        elif is_valid(year, num1):
            month, day = num1, num2
        else:
            day, month = num1, num2

        return pd.Timestamp(year=year, month=month, day=day)

    else:
        return pd.to_datetime(date_val, errors='coerce')

monitoring['Start_Date (Creation)'] = monitoring['Start_Date (Creation)'].apply(parse_mixed_date)
monitoring['Sample_Date'] = monitoring['Sample_Date'].apply(parse_mixed_date)

print(f"Date range: {monitoring['Sample_Date'].min()} to {monitoring['Sample_Date'].max()}")

## Compute day number

In [ ]:
monitoring['Day'] = (monitoring['Sample_Date'] - monitoring['Start_Date (Creation)']).dt.days

monitoring.head()

## Create tweet metadata table

In [ ]:
control.head()

In [ ]:
treatment.head()

In [ ]:
control_meta = control[['URL', 'User', 'Detection_Date', 'Creation_Date', 'Type', 'Narrative', 'Group']].copy()
control_meta['KPI'] = None
control_meta['Num_of_CNs'] = 0

treatment_meta = treatment[['URL', 'User', 'Detection_Date', 'Creation_Date', 'Type', 'Narrative', 'Group', 'KPI', 'Num_of_CNs']].copy()

tweet_meta = pd.concat([control_meta, treatment_meta], ignore_index=True)

# Compute detection lag (minutes between tweet creation and our detection)
tweet_meta['Detection_Date'] = pd.to_datetime(tweet_meta['Detection_Date'])
tweet_meta['Creation_Date'] = pd.to_datetime(tweet_meta['Creation_Date'])
tweet_meta['Detection_Lag_Minutes'] = (tweet_meta['Detection_Date'] - tweet_meta['Creation_Date']).dt.total_seconds() / 60

In [ ]:
tweet_meta.head()

## Pivot monitoring

In [ ]:
pivoted = monitoring.pivot_table(
    index='URL',
    columns='Day',
    values=['Views', 'Likes', 'Comments', 'Shares'],
    aggfunc='first'
)
pivoted.columns = [f'{metric}_Day{day}' for metric, day in pivoted.columns]
pivoted = pivoted.reset_index()

# Count monitoring days per tweet
monitoring_days = monitoring.groupby('URL')['Day'].nunique().reset_index()
monitoring_days.columns = ['URL', 'Num_Days']

In [ ]:
pivoted.head()

## Merge into analysis dataframe

In [ ]:
analysis_df = tweet_meta.merge(pivoted, on='URL', how='left')
analysis_df = analysis_df.merge(monitoring_days, on='URL', how='left')
analysis_df['Complete_Monitoring'] = analysis_df['Num_Days'] >= 14

analysis_df.head()

## Compute growth metrics

In [ ]:
for metric in ['Likes', 'Shares', 'Views', 'Comments']:
    day0 = f'{metric}_Day0'
    day13 = f'{metric}_Day13'

    # Percentage change: (final - initial) / (initial + 1) * 100
    # The +1 handles cases where initial = 0
    analysis_df[f'{metric}_Growth'] = (
        (analysis_df[day13] - analysis_df[day0]) / (analysis_df[day0] + 1) * 100
    )

## Summary

In [ ]:
complete_df = analysis_df[analysis_df['Complete_Monitoring']].copy()

print(f"\n=== DATA READY ===")
print(f"Total tweets: {len(analysis_df)}")
print(f"Complete 14-day monitoring: {len(complete_df)}")
print(f"  - Control: {len(complete_df[complete_df['Group'] == 'Control'])}")
print(f"  - Treatment: {len(complete_df[complete_df['Group'] == 'Treatment'])}")

# 2. Check Balance

**What "Balance" Means:**

At Day 0 (the moment we detected the tweet, before any CN effect could occur), Control and Treatment should look similar:

*   Similar engagement levels (Views, Likes, Shares, Comments)
*   Similar distribution across narratives
*   Similar distribution across tweet types (post vs comment)

**We'll use Mann-Whitney U test** (non-parametric, handles skewed distributions) to check if baseline engagement differs significantly. We want p > 0.05 (no significant difference).

In [ ]:
ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

## Baseline engagement (day 0)

In [ ]:
print("=== BASELINE ENGAGEMENT (DAY 0) ===\n")
print(f"{'Metric':<12} {'Control':>18} {'Treatment':>18} {'p-value':>12}")
print(f"{'':12} {'Mean (Median)':>18} {'Mean (Median)':>18} {'':>12}")
print("-" * 62)

for metric in ['Likes', 'Shares', 'Views', 'Comments']:
    col = f'{metric}_Day0'

    ctrl_mean = ctrl[col].mean()
    ctrl_med = ctrl[col].median()
    trt_mean = trt[col].mean()
    trt_med = trt[col].median()

    stat, p = stats.mannwhitneyu(ctrl[col].dropna(), trt[col].dropna())

    sig = "" if p > 0.05 else " *"
    print(f"{metric:<12} {ctrl_mean:>8.1f} ({ctrl_med:>5.0f}) {trt_mean:>8.1f} ({trt_med:>5.0f}) {p:>10.4f}{sig}")

print("\n* = significant difference (p < 0.05) — we want NO stars here")

## Narrative distribution

In [ ]:
print("\n\n=== NARRATIVE DISTRIBUTION ===\n")

narrative_balance = complete_df.groupby('Narrative')['Group'].value_counts().unstack(fill_value=0)
narrative_balance['Total'] = narrative_balance.sum(axis=1)
narrative_balance['Control %'] = (narrative_balance['Control'] / narrative_balance['Total'] * 100).round(1)
narrative_balance = narrative_balance.sort_values('Total', ascending=False)

print(narrative_balance.to_string())
print(f"\nIdeal split is 50%. Mean Control % = {narrative_balance['Control %'].mean():.1f}%")

## Tweet type distribution

In [ ]:
print("\n\n=== TWEET TYPE DISTRIBUTION ===\n")

type_balance = complete_df.groupby('Type')['Group'].value_counts().unstack(fill_value=0)
type_balance['Total'] = type_balance.sum(axis=1)
type_balance['Control %'] = (type_balance['Control'] / type_balance['Total'] * 100).round(1)

print(type_balance.to_string())

# 3. Main Effect Analysis

The core hypothesis of this experiment is that **counter-narratives (CNs) suppress the organic spread of harmful and misleading narrative tweets**. Concretely, we expect tweets that received CN replies (Treatment) to accumulate less engagement over time compared to tweets that were left alone (Control).

We focus on three engagement metrics: **Views**, **Likes**, and **Shares**.  
Comments are excluded from the main analysis because our CN intervention itself adds comment(s) to Treatment tweets, which mechanically inflates the Comments metric and confounds interpretation.

The analysis proceeds in 5 subsections, each answering a progressively deeper question:

| Section | Question |
|---------|----------|
| 3.1 | What does the raw suppression picture look like? |
| 3.2 | How large is the effect, and how certain are we? |
| 3.3 | When does suppression emerge, and how does it evolve? |
| 3.4 | Is the effect robust to baseline differences? |
| 3.5 | What does suppression look like at the individual tweet level? |

## 3.1 Descriptive Overview - The Suppression Story at a Glance

### Motivation

Before running any statistical tests, we need to clearly **see** the data. This subsection answers the most basic question: *"Did Treatment tweets grow less than Control tweets?"*

We compute engagement growth from Day 0 (detection) to multiple time horizons (1, 3, 7, and 13 days post-detection), and compare the two groups using both **medians** and **means**.

#### Why both medians and means?

Social media engagement data is notoriously **right-skewed** — most tweets get little engagement, while a few go semi-viral. This creates a situation where medians and means can tell very different stories:

- **Medians** reflect the *typical* tweet's experience. They are robust to outliers and answer: "What happened to the average tweet in each group?"

- **Means** are sensitive to the tails of the distribution. They answer: "What happened to the *total engagement* across each group?" — which matters because a single viral tweet can spread more harm than hundreds of low-engagement ones.

If CNs suppress the typical tweet (median effect) *and* prevent viral outliers (mean effect), that's the strongest evidence. If only one changes, that's still informative but tells a different story.

#### Why percentage growth (not absolute change)?

A tweet that goes from 10 to 20 views (+10) and a tweet that goes from 1000 to 1010 views (+10) had very different experiences. Percentage growth normalizes for baseline, making tweets comparable regardless of their initial engagement level.

We use the formula: `Growth% = (Day_N - Day_0) / (Day_0 + 1) × 100`  
The `+1` in the denominator prevents division by zero for tweets starting at 0 engagement, and has negligible impact on tweets with higher baselines.

#### Why these specific time windows?

- **1 day**: Captures immediate/short-term effect. Do CNs have an instant dampening effect?
- **3 days**: Twitter content has a short half-life; most engagement happens within 48-72 hours. This captures the "active spread" window.
- **7 days**: The medium-term. By this point, organic spread has largely plateaued for most tweets.
- **13 days** (full window): End-of-study cumulative effect. Note: we use Day 13 because our monitoring runs Day 0-13 (14 days inclusive).

In [ ]:
# ============================================================
# 3.1: DESCRIPTIVE OVERVIEW — THE SUPPRESSION STORY AT A GLANCE
# ============================================================

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Define analysis scope
METRICS = ['Views', 'Likes', 'Shares']  # Exclude Comments (CN adds comments to Treatment)
TIME_WINDOWS = {
    '1d':  ('Day0', 'Day1'),
    '3d':  ('Day0', 'Day3'),
    '7d':  ('Day0', 'Day7'),
    '13d': ('Day0', 'Day13')
}

# Ensure growth columns exist for our windows
for window_name, (start, end) in TIME_WINDOWS.items():
    for metric in METRICS:
        start_col = f'{metric}_{start}'
        end_col = f'{metric}_{end}'
        growth_col = f'{metric}_Growth_{window_name}'
        complete_df[growth_col] = (
            (complete_df[end_col] - complete_df[start_col]) /
            (complete_df[start_col] + 1) * 100
        )

# Split groups
ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

print(f"Analysis sample: Control n={len(ctrl)}, Treatment n={len(trt)}")
print(f"Metrics: {', '.join(METRICS)}")
print(f"Time windows: {', '.join(TIME_WINDOWS.keys())}")

### 3.1.1 Summary Table — Median Growth Comparison

This is the primary summary table. For each metric and time window, we show:
- **Control Median Growth%**: How much the typical Control tweet grew
- **Treatment Median Growth%**: How much the typical Treatment tweet grew
- **Absolute Gap**: Control median − Treatment median (positive = suppression)
- **Relative Suppression%**: What percentage less the Treatment group grew relative to Control

A positive "Relative Suppression%" means Treatment tweets grew less — which supports our hypothesis.
A negative value means Treatment actually grew *more*, which would contradict it.

In [ ]:
# --------------------------------------------------
# 3.1.1: MEDIAN GROWTH COMPARISON TABLE
# --------------------------------------------------

print("=" * 90)
print("TABLE 1: MEDIAN ENGAGEMENT GROWTH — CONTROL vs TREATMENT")
print("=" * 90)
print(f"\n{'Metric':<10} {'Window':<8} {'Ctrl Med%':>12} {'Trt Med%':>12} {'Abs Gap':>12} {'Relative Supp%':>16}")
print("-" * 72)

summary_rows = []

for metric in METRICS:
    for window_name in TIME_WINDOWS.keys():
        col = f'{metric}_Growth_{window_name}'

        ctrl_med = ctrl[col].median()
        trt_med = trt[col].median()
        abs_gap = ctrl_med - trt_med

        # Relative suppression: how much less did Treatment grow compared to Control?
        # Only meaningful when Control median > 0
        if ctrl_med > 0:
            rel_supp = (abs_gap / ctrl_med) * 100
            rel_str = f"{rel_supp:>+.1f}%"
        elif ctrl_med == 0 and trt_med == 0:
            rel_supp = 0
            rel_str = "both 0%"
        else:
            rel_supp = None
            rel_str = "N/A"

        summary_rows.append({
            'Metric': metric, 'Window': window_name,
            'Ctrl_Median': ctrl_med, 'Trt_Median': trt_med,
            'Abs_Gap': abs_gap, 'Rel_Suppression': rel_supp
        })

        print(f"{metric:<10} {window_name:<8} {ctrl_med:>+11.1f}% {trt_med:>+11.1f}% {abs_gap:>+11.1f} {rel_str:>16}")
    print("-" * 72)

summary_df = pd.DataFrame(summary_rows)

### 3.1.2 Summary Table — Mean Growth Comparison

Mean-based comparison captures the influence of outlier tweets — the ones that matter most for narrative spread. Because social media engagement follows a heavy-tailed distribution, a few tweets with explosive growth can dominate the mean while leaving the median unchanged.

**Interpreting mean vs median divergence:**
- If means differ dramatically but medians don't → the CN effect is concentrated in the tails (CNs prevent viral breakouts but don't affect typical tweets much)
- If both means and medians show the same pattern → the CN effect is broad-based
- If medians differ but means don't → the effect is on typical tweets, but outliers wash it out

We also report the **mean absolute change** (not just percentage) to give a concrete sense of how many views/likes/shares were prevented.

In [ ]:
# --------------------------------------------------
# 3.1.2: MEAN GROWTH COMPARISON TABLE
# --------------------------------------------------

print("=" * 90)
print("TABLE 2: MEAN ENGAGEMENT GROWTH — CONTROL vs TREATMENT")
print("=" * 90)
print(f"\n{'Metric':<10} {'Window':<8} {'Ctrl Mean%':>13} {'Trt Mean%':>13} {'Abs Gap':>12} {'Relative Supp%':>16}")
print("-" * 74)

for metric in METRICS:
    for window_name, (start, end) in TIME_WINDOWS.items():
        col = f'{metric}_Growth_{window_name}'

        ctrl_mean = ctrl[col].mean()
        trt_mean = trt[col].mean()
        abs_gap = ctrl_mean - trt_mean

        if ctrl_mean > 0:
            rel_supp = (abs_gap / ctrl_mean) * 100
            rel_str = f"{rel_supp:>+.1f}%"
        else:
            rel_str = "N/A"

        print(f"{metric:<10} {window_name:<8} {ctrl_mean:>+12.1f}% {trt_mean:>+12.1f}% {abs_gap:>+11.1f} {rel_str:>16}")
    print("-" * 74)

# Also show mean ABSOLUTE change (not percentage)
print("\n\n" + "=" * 90)
print("TABLE 3: MEAN ABSOLUTE ENGAGEMENT CHANGE (raw units, not %)")
print("=" * 90)
print(f"\n{'Metric':<10} {'Window':<8} {'Ctrl Mean Δ':>14} {'Trt Mean Δ':>14} {'Difference':>14}")
print("-" * 62)

for metric in METRICS:
    for window_name, (start, end) in TIME_WINDOWS.items():
        start_col = f'{metric}_{start}'
        end_col = f'{metric}_{end}'

        ctrl_abs = (ctrl[end_col] - ctrl[start_col]).mean()
        trt_abs = (trt[end_col] - trt[start_col]).mean()
        diff = ctrl_abs - trt_abs

        print(f"{metric:<10} {window_name:<8} {ctrl_abs:>+13.1f} {trt_abs:>+13.1f} {diff:>+13.1f}")
    print("-" * 62)

### 3.1.3 Distribution Shape Overview

Before interpreting the numbers above, we need to understand the **shape** of our data. This is critical because:

1. **Skewness** determines whether means or medians are more trustworthy
2. **Zero-inflation** (many tweets with 0 growth) affects which statistical tests are appropriate
3. **Outlier prevalence** tells us whether the mean is being driven by a handful of extreme cases

For each metric, we report: skewness, kurtosis, the % of tweets with exactly 0 growth, and the 1st/99th percentiles to characterize the tails.

In [ ]:
# --------------------------------------------------
# 3.1.3: DISTRIBUTION SHAPE DIAGNOSTICS
# --------------------------------------------------

print("=" * 110)
print("TABLE 4: DISTRIBUTION SHAPE DIAGNOSTICS (13-day growth)")
print("=" * 110)
print(f"\n{'Metric':<10} {'Group':<12} {'Skewness':>10} {'Kurtosis':>10} {'Zero%':>8} {'P1':>10} {'P25':>10} {'P50':>10} {'P75':>10} {'P99':>10}")
print("-" * 102)

for metric in METRICS:
    col = f'{metric}_Growth_13d'
    for group_name, group_df in [('Control', ctrl), ('Treatment', trt)]:
        vals = group_df[col].dropna()
        skew = vals.skew()
        kurt = vals.kurtosis()
        zero_pct = (vals == 0).mean() * 100
        p1, p25, p50, p75, p99 = np.percentile(vals, [1, 25, 50, 75, 99])

        print(f"{metric:<10} {group_name:<12} {skew:>10.1f} {kurt:>10.1f} {zero_pct:>7.1f}% {p1:>10.1f} {p25:>10.1f} {p50:>10.1f} {p75:>10.1f} {p99:>10.1f}")
    print()

### 3.1.4: Visual Overview — Bar Chart with Medians

A grouped bar chart showing median growth for Control vs Treatment across all metrics and time windows. This is the "one figure that summarizes the story" — the kind of figure that would appear early in a paper to give the reader an immediate visual sense of the effect.

We use medians here because they represent the typical tweet's experience and are not distorted by outliers.

In [ ]:
# --------------------------------------------------
# 3.1.4: VISUAL OVERVIEW — GROUPED BAR CHART (MEDIANS)
# --------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Median Engagement Growth: Control vs Treatment', fontsize=14, fontweight='bold', y=1.02)

windows = list(TIME_WINDOWS.keys())
x = np.arange(len(windows))
width = 0.35

for idx, metric in enumerate(METRICS):
    ax = axes[idx]

    ctrl_medians = [ctrl[f'{metric}_Growth_{w}'].median() for w in windows]
    trt_medians = [trt[f'{metric}_Growth_{w}'].median() for w in windows]

    bars_ctrl = ax.bar(x - width/2, ctrl_medians, width, label='Control', color='#4C72B0', alpha=0.85)
    bars_trt = ax.bar(x + width/2, trt_medians, width, label='Treatment', color='#DD8452', alpha=0.85)

    # Add value labels on bars
    for bars in [bars_ctrl, bars_trt]:
        for bar in bars:
            height = bar.get_height()
            if height != 0:
                ax.annotate(f'{height:.0f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

    ax.set_xlabel('Time Window')
    ax.set_ylabel('Median Growth %')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(windows)
    ax.legend()
    ax.axhline(0, color='black', linewidth=0.5, linestyle='-')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_1_median_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.1.5: Visual Overview — Mean Comparison with Outlier Context

Since the means can be heavily influenced by outliers, we present two complementary views:

1. **Standard mean comparison** (bar chart) — shows the raw mean difference, including outlier effects
2. **Trimmed mean comparison** (5% trimmed) — removes the top and bottom 5% of values from each group before computing the mean, giving a more robust central tendency that's still sensitive to more of the distribution than the median

The trimmed mean is a useful "middle ground" between the median (ignores everything but the center) and the full mean (dominated by extremes). If the trimmed mean tells the same story as the full mean, outliers aren't driving the result. If it diverges, we know the effect is concentrated in the tails.

In [ ]:
# --------------------------------------------------
# 3.1.5: MEAN AND TRIMMED MEAN COMPARISON
# --------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Mean Engagement Growth: Full Mean vs 5% Trimmed Mean', fontsize=14, fontweight='bold', y=1.02)

windows = list(TIME_WINDOWS.keys())
x = np.arange(len(windows))
width = 0.35

for idx, metric in enumerate(METRICS):
    for row, (label, mean_func) in enumerate([
        ('Full Mean', lambda x: x.mean()),
        ('5% Trimmed Mean', lambda x: stats.trim_mean(x.dropna(), 0.05))
    ]):
        ax = axes[row, idx]

        ctrl_means = [mean_func(ctrl[f'{metric}_Growth_{w}']) for w in windows]
        trt_means = [mean_func(trt[f'{metric}_Growth_{w}']) for w in windows]

        bars_ctrl = ax.bar(x - width/2, ctrl_means, width, label='Control', color='#4C72B0', alpha=0.85)
        bars_trt = ax.bar(x + width/2, trt_means, width, label='Treatment', color='#DD8452', alpha=0.85)

        for bars in [bars_ctrl, bars_trt]:
            for bar in bars:
                height = bar.get_height()
                if abs(height) > 0.5:
                    ax.annotate(f'{height:.0f}%',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=7)

        ax.set_xlabel('Time Window')
        ax.set_ylabel(f'{label} Growth %')
        ax.set_title(f'{metric} — {label}', fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(windows)
        ax.legend(fontsize=9)
        ax.axhline(0, color='black', linewidth=0.5, linestyle='-')
        ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_1_mean_trimmed.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.1.6: Outlier Audit

Given the extreme skewness of social media data, it's important to explicitly identify and quantify outliers. This serves two purposes:

1. **Transparency**: Readers (and reviewers) need to know how many extreme values exist and how much they influence the results
2. **Robustness planning**: If outliers are driving the effect, we need to check that our conclusions hold without them (which we'll do formally in Section 3.4)

We define outliers using the **IQR method** (values beyond Q1 − 1.5×IQR or Q3 + 1.5×IQR), which is distribution-agnostic and commonly used.

In [ ]:
# --------------------------------------------------
# 3.1.6: OUTLIER AUDIT
# --------------------------------------------------

print("=" * 90)
print("TABLE 5: OUTLIER AUDIT (13-day growth, IQR method)")
print("=" * 90)
print(f"\n{'Metric':<10} {'Group':<12} {'n':>6} {'Outliers':>10} {'Outlier%':>10} {'Max Growth':>12} {'Mean w/o OL':>14}")
print("-" * 76)

for metric in METRICS:
    col = f'{metric}_Growth_13d'
    for group_name, group_df in [('Control', ctrl), ('Treatment', trt)]:
        vals = group_df[col].dropna()
        q1 = vals.quantile(0.25)
        q3 = vals.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outliers = vals[(vals < lower) | (vals > upper)]
        non_outliers = vals[(vals >= lower) & (vals <= upper)]

        print(f"{metric:<10} {group_name:<12} {len(vals):>6} {len(outliers):>10} {len(outliers)/len(vals)*100:>9.1f}% {vals.max():>11.0f}% {non_outliers.mean():>13.1f}%")
    print()

### 3.1.7: Key Takeaways from Section 3.1

This cell programmatically summarizes the main findings from the descriptive overview. It identifies which metrics show clear suppression patterns and flags any anomalies that need attention in subsequent analyses.


In [ ]:
# --------------------------------------------------
# 3.1.7: AUTOMATED KEY TAKEAWAYS
# --------------------------------------------------

print("=" * 70)
print("SECTION 3.1 — KEY TAKEAWAYS")
print("=" * 70)

for metric in METRICS:
    col_13d = f'{metric}_Growth_13d'
    ctrl_med = ctrl[col_13d].median()
    trt_med = trt[col_13d].median()
    ctrl_mean = ctrl[col_13d].mean()
    trt_mean = trt[col_13d].mean()
    ctrl_tmean = stats.trim_mean(ctrl[col_13d].dropna(), 0.05)
    trt_tmean = stats.trim_mean(trt[col_13d].dropna(), 0.05)

    print(f"\n{'─' * 60}")
    print(f"  {metric.upper()}")
    print(f"{'─' * 60}")

    # Median comparison
    if ctrl_med == 0 and trt_med == 0:
        print(f"  Median: Both groups have 0% median growth.")
        print(f"    → Most tweets gain no {metric.lower()}; median comparison is uninformative.")
        print(f"    → Must rely on mean/distributional analysis for this metric.")
    elif ctrl_med > trt_med:
        gap = ((ctrl_med - trt_med) / ctrl_med * 100) if ctrl_med != 0 else 0
        print(f"  Median: Control ({ctrl_med:.1f}%) > Treatment ({trt_med:.1f}%)")
        print(f"    → Typical Treatment tweet grew {gap:.1f}% less — SUPPORTS suppression.")
    else:
        print(f"  Median: Control ({ctrl_med:.1f}%) < Treatment ({trt_med:.1f}%)")
        print(f"    → Treatment median is HIGHER — does NOT support suppression at the median.")
        print(f"    → This may reflect a distributional difference (see Sections 3.2–3.5).")

    # Mean comparison
    if ctrl_mean > trt_mean:
        gap_mean = ((ctrl_mean - trt_mean) / ctrl_mean * 100) if ctrl_mean != 0 else 0
        print(f"  Mean: Control ({ctrl_mean:.1f}%) > Treatment ({trt_mean:.1f}%)")
        print(f"    → Total engagement growth {gap_mean:.1f}% lower in Treatment — SUPPORTS suppression.")
    else:
        print(f"  Mean: Control ({ctrl_mean:.1f}%) < Treatment ({trt_mean:.1f}%)")
        print(f"    → Treatment mean is HIGHER — does NOT support suppression in the mean.")

    # Trimmed mean
    if ctrl_tmean > trt_tmean:
        print(f"  Trimmed Mean (5%): Control ({ctrl_tmean:.1f}%) > Treatment ({trt_tmean:.1f}%)")
        print(f"    → Suppression holds after removing extreme tails.")
    else:
        print(f"  Trimmed Mean (5%): Control ({ctrl_tmean:.1f}%) < Treatment ({trt_tmean:.1f}%)")
        print(f"    → Effect reverses without extremes — outliers may be driving the mean result.")

    # Median vs Mean divergence flag
    if (ctrl_med > trt_med) != (ctrl_mean > trt_mean):
        print(f"  ⚠ WARNING: Median and mean tell OPPOSITE stories for {metric}!")
        print(f"    → Effect size analysis (Section 3.2) is critical for this metric.")

print(f"\n{'═' * 70}")
print("END OF SECTION 3.1")
print(f"{'═' * 70}")

## 3.2: Effect Size Quantification

### Motivation

Section 3.1 showed us the *descriptive* picture: means suggest suppression, medians are ambiguous (zero-inflated for Likes/Shares, reversed for Views). But descriptive summaries alone can't answer two critical questions:

1. **How large is the effect?** — P-values (which we'll compute here) tell us *whether* the groups differ, but not *how much*. A p < 0.001 with a tiny effect size is scientifically uninteresting. A moderate effect size with p = 0.08 might still be practically meaningful. We need proper effect size measures.

2. **How certain are we?** — Point estimates (medians, means) are just single numbers. We need **confidence intervals** to know the range of plausible effect sizes. A paper that says "Treatment reduced engagement by 15–40% (95% CI)" is far more compelling than "p = 0.03."

### What we compute

For each metric × time window, we run three complementary analyses:

#### A. Mann-Whitney U Test with Rank-Biserial Correlation

The **Mann-Whitney U test** is the standard non-parametric test for comparing two independent groups. It tests whether one group tends to have larger values than the other — formally, whether P(Control > Treatment) ≠ 0.5. It makes no assumptions about distribution shape, which is essential given our extreme skewness.

The natural effect size for Mann-Whitney is the **rank-biserial correlation (r)**:

`r = 1 − (2U) / (n₁ × n₂)`

This ranges from −1 to +1 and has a direct probabilistic interpretation:
- r = 0 → a random Control tweet and a random Treatment tweet are equally likely to have higher growth
- r = +0.3 → a random Control tweet has higher growth about 65% of the time (suppression)
- r = −0.3 → a random Treatment tweet has higher growth about 65% of the time (anti-suppression)

Standard benchmarks (Cohen): |r| < 0.1 = negligible, 0.1–0.3 = small, 0.3–0.5 = medium, > 0.5 = large.

#### B. Brunner-Munzel Test

While Mann-Whitney is well-known, it technically tests whether the two distributions are *stochastically ordered* (one consistently larger). When distributions have very different shapes (which ours likely do given the skewness differences), the **Brunner-Munzel test** is more appropriate. It directly estimates the **stochastic superiority probability** P(Control > Treatment) without assuming equal variance or shape.

We include this because reviewers familiar with non-parametric methods may note that Mann-Whitney's validity can be affected by heteroscedasticity, and Brunner-Munzel addresses exactly that concern.

#### C. Bootstrapped Confidence Intervals

We compute **95% bootstrap CIs** (10,000 resamples) for three quantities:
- The **median difference** (Control median − Treatment median): the typical-tweet effect
- The **trimmed mean difference** (5% trimmed): a robust central tendency effect
- The **rank-biserial r**: uncertainty around the effect size itself

Bootstrapping is distribution-free and works well for skewed data where parametric CIs would be inappropriate.

#### D. Permutation Test

As a final robustness check, we run a **permutation test** on the median difference. Unlike Mann-Whitney (which tests rank ordering), the permutation test directly tests whether the *observed* median difference could have arisen by chance under random assignment. It is the most assumption-free test available — it only assumes exchangeability of group labels, which is guaranteed by our randomized design.

### Why four tests and not just one?

Each test answers a slightly different question and has different strengths:

| Test | Tests for | Strengths | Limitations |
|------|-----------|-----------|-------------|
| Mann-Whitney U | Stochastic ordering | Standard, well-known, provides effect size (r) | Assumes equal shapes |
| Brunner-Munzel | P(X > Y) ≠ 0.5 | Handles unequal shapes/variances | Less commonly known |
| Bootstrap CI | Range of plausible effects | No distributional assumptions, highly flexible | Computationally intensive |
| Permutation test | Observed difference under null | Exact, assumption-free | Only tests one statistic at a time |

If all four agree, we have very strong evidence. If they diverge, the pattern of agreement/disagreement itself is informative.

### 3.2.1: Statistical Tests — Full Results Table

For each metric and time window, we report:
- Mann-Whitney U p-value and rank-biserial r (with interpretation)
- Brunner-Munzel p-value and P(Control > Treatment) estimate
- Permutation test p-value for the median difference

In [ ]:
# --------------------------------------------------
# 3.2.1: STATISTICAL TESTS — FULL RESULTS TABLE
# --------------------------------------------------

from scipy import stats
import numpy as np

np.random.seed(42)  # Reproducibility for bootstrap/permutation

METRICS = ['Views', 'Likes', 'Shares']
TIME_WINDOWS = {
    '1d':  ('Day0', 'Day1'),
    '3d':  ('Day0', 'Day3'),
    '7d':  ('Day0', 'Day7'),
    '13d': ('Day0', 'Day13')
}

# Store all results for later use
effect_results = []

print("=" * 120)
print("TABLE 6: COMPREHENSIVE STATISTICAL TESTS — CONTROL vs TREATMENT GROWTH")
print("=" * 120)

for metric in METRICS:
    print(f"\n{'━' * 120}")
    print(f"  {metric.upper()}")
    print(f"{'━' * 120}")
    print(f"  {'Window':<8} │ {'MWU p':>10} {'r (rank-bis)':>14} {'Interpret':>10} │ {'BM p':>10} {'P(C>T)':>10} │ {'Perm p':>10}")
    print(f"  {'─' * 8}─┼─{'─' * 10}─{'─' * 14}─{'─' * 10}─┼─{'─' * 10}─{'─' * 10}─┼─{'─' * 10}")

    for window_name in TIME_WINDOWS.keys():
        col = f'{metric}_Growth_{window_name}'
        ctrl_vals = ctrl[col].dropna().values
        trt_vals = trt[col].dropna().values

        # --- Mann-Whitney U with rank-biserial ---
        mwu_stat, mwu_p = stats.mannwhitneyu(ctrl_vals, trt_vals, alternative='two-sided')
        n1, n2 = len(ctrl_vals), len(trt_vals)
        # scipy's mannwhitneyu(x,y) returns U_x; large U_x means x tends to be larger
        # r = 1 - 2U/(n1*n2) gives NEGATIVE r when first group (ctrl) is larger
        # We FLIP the sign so that POSITIVE r = ctrl grew more = suppression
        rank_biserial = (2 * mwu_stat) / (n1 * n2) - 1

        # Interpret effect size
        abs_r = abs(rank_biserial)
        if abs_r < 0.1:
            r_interp = "negligible"
        elif abs_r < 0.3:
            r_interp = "small"
        elif abs_r < 0.5:
            r_interp = "medium"
        else:
            r_interp = "large"

        # --- Brunner-Munzel ---
        bm_stat, bm_p = stats.brunnermunzel(ctrl_vals, trt_vals)
        # P(Control > Treatment) = estimated from BM
        # The BM statistic relates to the probability; we can estimate it directly
        # P(Control > Treatment) derived from Brunner-Munzel framework:
        # Compute from ranks of the combined sample — this is the quantity BM tests
        combined = np.concatenate([ctrl_vals, trt_vals])
        ranks = stats.rankdata(combined)
        ranks_ctrl = ranks[:n1]
        p_ctrl_gt_trt = (np.mean(ranks_ctrl) - (n1 + 1) / 2) / n2

        # --- Permutation test on median difference ---
        def median_diff(x, y):
            return np.median(x) - np.median(y)

        perm_res = stats.permutation_test(
            (ctrl_vals, trt_vals),
            lambda x, y, axis=None: np.array([np.median(x) - np.median(y)]),
            n_resamples=9999, random_state=42
        )
        perm_p = perm_res.pvalue

        # Store results
        effect_results.append({
            'Metric': metric, 'Window': window_name,
            'MWU_p': mwu_p, 'Rank_Biserial': rank_biserial, 'R_Interpret': r_interp,
            'BM_p': bm_p, 'P_Ctrl_GT_Trt': p_ctrl_gt_trt,
            'Perm_p': perm_p, 'Median_Diff': perm_res.statistic
        })

        # Format significance markers
        def sig_marker(p):
            if p < 0.001: return "***"
            elif p < 0.01: return "** "
            elif p < 0.05: return "*  "
            elif p < 0.1: return "†  "
            else: return "   "

        print(f"  {window_name:<8} │ {mwu_p:>9.4f}{sig_marker(mwu_p)} {rank_biserial:>+10.4f}    {r_interp:>10} │ {bm_p:>9.4f}{sig_marker(bm_p)} {p_ctrl_gt_trt:>9.3f} │ {perm_p:>9.4f}{sig_marker(perm_p)}")

print(f"\n{'─' * 120}")
print("  Significance: *** p<0.001, ** p<0.01, * p<0.05, † p<0.1")
print("  Rank-biserial r: +ve = Control grew MORE (suppression), −ve = Treatment grew MORE")
print("  P(C>T): Probability that a random Control tweet grew more than a random Treatment tweet")

effect_df = pd.DataFrame(effect_results)

### 3.2.2: Bootstrapped Confidence Intervals

The point estimates above tell us the *best guess* for the effect size, but we need to know how precise that guess is. Bootstrapping resamples our data 10,000 times to estimate the sampling distribution of each statistic.

We compute 95% CIs for three quantities:
1. **Median difference** (Control − Treatment): interpretable in original units (% growth)
2. **5% Trimmed mean difference**: more sensitive to distributional shifts than the median
3. **Rank-biserial r**: the probability-based effect size from the Mann-Whitney test

A CI that **excludes zero** means the effect is statistically significant at α = 0.05. A CI that is **narrow** means we have precise estimates. Both matter.

In [ ]:
# --------------------------------------------------
# 3.2.2: BOOTSTRAPPED CONFIDENCE INTERVALS
# --------------------------------------------------

np.random.seed(42)
N_BOOTSTRAP = 10000

print("=" * 115)
print("TABLE 7: BOOTSTRAPPED 95% CONFIDENCE INTERVALS")
print("=" * 115)

bootstrap_results = []

for metric in METRICS:
    print(f"\n{'━' * 115}")
    print(f"  {metric.upper()}")
    print(f"{'━' * 115}")
    print(f"  {'Window':<8} │ {'Median Diff (C−T)':>22} {'95% CI':>22} │ {'Trim Mean Diff':>18} {'95% CI':>22} │ {'Excl 0?':>8}")
    print(f"  {'─' * 8}─┼─{'─' * 22}─{'─' * 22}─┼─{'─' * 18}─{'─' * 22}─┼─{'─' * 8}")

    for window_name in TIME_WINDOWS.keys():
        col = f'{metric}_Growth_{window_name}'
        ctrl_vals = ctrl[col].dropna().values
        trt_vals = trt[col].dropna().values

        # --- Bootstrap median difference ---
        median_diffs = []
        tmean_diffs = []
        r_vals = []

        for _ in range(N_BOOTSTRAP):
            c_sample = np.random.choice(ctrl_vals, size=len(ctrl_vals), replace=True)
            t_sample = np.random.choice(trt_vals, size=len(trt_vals), replace=True)

            median_diffs.append(np.median(c_sample) - np.median(t_sample))
            tmean_diffs.append(
                stats.trim_mean(c_sample, 0.05) - stats.trim_mean(t_sample, 0.05)
            )

        median_diffs = np.array(median_diffs)
        tmean_diffs = np.array(tmean_diffs)

        # 95% CIs (percentile method)
        med_ci_lo, med_ci_hi = np.percentile(median_diffs, [2.5, 97.5])
        tm_ci_lo, tm_ci_hi = np.percentile(tmean_diffs, [2.5, 97.5])

        med_point = np.median(ctrl_vals) - np.median(trt_vals)
        tm_point = stats.trim_mean(ctrl_vals, 0.05) - stats.trim_mean(trt_vals, 0.05)

        # Does CI exclude zero?
        med_excludes_zero = (med_ci_lo > 0 or med_ci_hi < 0)
        tm_excludes_zero = (tm_ci_lo > 0 or tm_ci_hi < 0)
        either_excludes = "YES" if (med_excludes_zero or tm_excludes_zero) else "no"

        bootstrap_results.append({
            'Metric': metric, 'Window': window_name,
            'Median_Diff': med_point, 'Median_CI_Lo': med_ci_lo, 'Median_CI_Hi': med_ci_hi,
            'TMean_Diff': tm_point, 'TMean_CI_Lo': tm_ci_lo, 'TMean_CI_Hi': tm_ci_hi,
            'Med_Excludes_Zero': med_excludes_zero, 'TM_Excludes_Zero': tm_excludes_zero
        })

        print(f"  {window_name:<8} │ {med_point:>+18.1f}pp   [{med_ci_lo:>+8.1f}, {med_ci_hi:>+8.1f}] │ {tm_point:>+14.1f}pp   [{tm_ci_lo:>+8.1f}, {tm_ci_hi:>+8.1f}] │ {either_excludes:>8}")

print(f"\n{'─' * 115}")
print("  'pp' = percentage points difference in growth rate")
print("  CI excluding zero → statistically significant at α=0.05")

bootstrap_df = pd.DataFrame(bootstrap_results)

### 3.2.3: Effect Size Forest Plot

A **forest plot** is the standard visualization for effect sizes with confidence intervals. Each row shows one metric×window combination, with the point estimate as a dot and the 95% CI as a horizontal line. The vertical dashed line at zero represents "no effect."

Points to the right of zero indicate Control grew more (suppression). Points to the left indicate Treatment grew more. If the CI crosses zero, the effect is not statistically significant.

We show two panels:
1. **Rank-biserial r** — the probability-based effect size (standardized, comparable across metrics)
2. **Trimmed mean difference** — the effect in interpretable units (percentage points of growth)

In [ ]:
# --------------------------------------------------
# 3.2.3: EFFECT SIZE FOREST PLOT
# --------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# --- Panel A: Rank-Biserial r ---
ax = axes[0]
labels = []
positions = []
pos = 0

for metric in METRICS:
    for window_name in TIME_WINDOWS.keys():
        row = effect_df[(effect_df['Metric'] == metric) & (effect_df['Window'] == window_name)].iloc[0]
        r = row['Rank_Biserial']
        p = row['MWU_p']

        color = '#4C72B0' if p < 0.05 else '#999999'
        marker = 'D' if p < 0.05 else 'o'

        ax.plot(r, pos, marker=marker, color=color, markersize=8, zorder=5)
        labels.append(f"{metric} ({window_name})")
        positions.append(pos)
        pos += 1
    pos += 0.5  # Gap between metrics

ax.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax.set_yticks(positions)
ax.set_yticklabels(labels)
ax.set_xlabel('Rank-Biserial r\n(+ve = Control grew MORE = suppression)', fontsize=10)
ax.set_title('Panel A: Rank-Biserial Effect Size', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

# Add interpretation bands
for threshold, label, alpha in [(0.1, 'small', 0.05), (0.3, 'medium', 0.05)]:
    ax.axvspan(threshold, threshold + 0.001, alpha=0, color='gray')
    ax.axvspan(-threshold, -threshold - 0.001, alpha=0, color='gray')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='D', color='#4C72B0', linestyle='None', markersize=8, label='p < 0.05'),
    Line2D([0], [0], marker='o', color='#999999', linestyle='None', markersize=8, label='p ≥ 0.05')
]
ax.legend(handles=legend_elements, loc='lower right')

# --- Panel B: Trimmed Mean Difference with Bootstrap CIs ---
ax = axes[1]
positions = []
pos = 0

for metric in METRICS:
    for window_name in TIME_WINDOWS.keys():
        row = bootstrap_df[(bootstrap_df['Metric'] == metric) & (bootstrap_df['Window'] == window_name)].iloc[0]

        point = row['TMean_Diff']
        ci_lo = row['TMean_CI_Lo']
        ci_hi = row['TMean_CI_Hi']
        excludes_zero = row['TM_Excludes_Zero']

        color = '#C44E52' if excludes_zero else '#999999'

        ax.errorbar(point, pos, xerr=[[point - ci_lo], [ci_hi - point]],
                    fmt='D' if excludes_zero else 'o', color=color,
                    capsize=4, capthick=1.5, markersize=7, zorder=5)
        positions.append(pos)
        pos += 1
    pos += 0.5

ax.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax.set_yticks(positions)
ax.set_yticklabels([f"{m} ({w})" for m in METRICS for w in TIME_WINDOWS.keys()])
ax.set_xlabel('Trimmed Mean Difference (C−T) in percentage points\n(+ve = suppression)', fontsize=10)
ax.set_title('Panel B: Trimmed Mean Difference with 95% Bootstrap CI', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

legend_elements = [
    Line2D([0], [0], marker='D', color='#C44E52', linestyle='None', markersize=7, label='CI excludes 0'),
    Line2D([0], [0], marker='o', color='#999999', linestyle='None', markersize=7, label='CI includes 0')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.suptitle('Effect Size Summary: Counter-Narrative Suppression of Engagement Growth',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_2_forest_plot.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2.4: Cross-Test Agreement Heatmap

With four different statistical tests, we need a clear way to see whether they **agree or disagree**. This heatmap shows, for each metric × window, how many tests found a significant result (at α = 0.05) and in which direction.

Strong evidence requires **convergence across methods**. If all four tests agree, we can be confident. If they diverge, the specific pattern tells us what's happening (e.g., "significant by rank tests but not by median tests" suggests the effect is in the tails, not the center).

In [ ]:
# --------------------------------------------------
# 3.2.4: CROSS-TEST AGREEMENT HEATMAP
# --------------------------------------------------

# Build agreement matrix
agreement_data = []

for metric in METRICS:
    for window_name in TIME_WINDOWS.keys():
        e_row = effect_df[(effect_df['Metric'] == metric) & (effect_df['Window'] == window_name)].iloc[0]
        b_row = bootstrap_df[(bootstrap_df['Metric'] == metric) & (bootstrap_df['Window'] == window_name)].iloc[0]

        # Direction: positive rank-biserial = Control grew more = suppression
        direction = "suppression" if e_row['Rank_Biserial'] > 0 else "anti-suppression"

        tests = {
            'Mann-Whitney U': e_row['MWU_p'] < 0.05,
            'Brunner-Munzel': e_row['BM_p'] < 0.05,
            'Permutation': e_row['Perm_p'] < 0.05,
            'Bootstrap CI\n(Trim Mean)': b_row['TM_Excludes_Zero']
        }

        n_sig = sum(tests.values())

        agreement_data.append({
            'Metric': metric, 'Window': window_name,
            'Direction': direction, 'N_Significant': n_sig,
            **{k: 1 if v else 0 for k, v in tests.items()}
        })

agreement_df = pd.DataFrame(agreement_data)

# Create heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Reshape for heatmap: rows = metric×window, cols = tests
row_labels = [f"{r['Metric']} ({r['Window']})" for _, r in agreement_df.iterrows()]
test_cols = ['Mann-Whitney U', 'Brunner-Munzel', 'Permutation', 'Bootstrap CI\n(Trim Mean)']
heatmap_data = agreement_df[['Mann-Whitney U', 'Brunner-Munzel', 'Permutation', 'Bootstrap CI\n(Trim Mean)']].values

# Custom colormap: 0 = light gray, 1 = colored by direction
import matplotlib.colors as mcolors

# Create colored matrix: green for significant suppression, red for significant anti-suppression, gray for non-sig
color_matrix = np.zeros((*heatmap_data.shape, 3))
for i in range(heatmap_data.shape[0]):
    direction = agreement_df.iloc[i]['Direction']
    for j in range(heatmap_data.shape[1]):
        if heatmap_data[i, j] == 1:
            if direction == 'suppression':
                color_matrix[i, j] = [0.2, 0.7, 0.3]  # Green = significant + suppression
            else:
                color_matrix[i, j] = [0.9, 0.3, 0.3]  # Red = significant + anti-suppression
        else:
            color_matrix[i, j] = [0.9, 0.9, 0.9]  # Light gray = not significant

ax.imshow(color_matrix, aspect='auto')

# Add text labels
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        text = "SIG" if heatmap_data[i, j] == 1 else "n.s."
        color = 'white' if heatmap_data[i, j] == 1 else 'gray'
        ax.text(j, i, text, ha='center', va='center', fontsize=10, fontweight='bold', color=color)

# Add N_Significant summary column
for i, row in agreement_df.iterrows():
    n = row['N_Significant']
    ax.text(len(test_cols) + 0.3, i, f"{n}/4", ha='center', va='center', fontsize=11, fontweight='bold',
            color='darkgreen' if n >= 3 else ('orange' if n >= 1 else 'gray'))

ax.set_xticks(range(len(test_cols)))
ax.set_xticklabels(test_cols, fontsize=10)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=10)
ax.set_title('Cross-Test Agreement: Which Tests Find Significant Suppression?', fontsize=13, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=[0.2, 0.7, 0.3], label='Significant + Suppression direction'),
    Patch(facecolor=[0.9, 0.3, 0.3], label='Significant + Anti-suppression direction'),
    Patch(facecolor=[0.9, 0.9, 0.9], edgecolor='gray', label='Not significant')
]
ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0, -0.08), ncol=3, fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_2_agreement_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2.5: Key Takeaways from Section 3.2

Programmatic summary of all effect size findings, cross-referencing across tests to produce a robust conclusion for each metric.

In [ ]:
# --------------------------------------------------
# 3.2.5: AUTOMATED KEY TAKEAWAYS
# --------------------------------------------------

print("=" * 75)
print("SECTION 3.2 — KEY TAKEAWAYS")
print("=" * 75)

for metric in METRICS:
    print(f"\n{'─' * 65}")
    print(f"  {metric.upper()}")
    print(f"{'─' * 65}")

    # Gather results across all windows
    metric_effects = effect_df[effect_df['Metric'] == metric]
    metric_bootstrap = bootstrap_df[bootstrap_df['Metric'] == metric]

    # 13-day results (primary endpoint)
    e13 = metric_effects[metric_effects['Window'] == '13d'].iloc[0]
    b13 = metric_bootstrap[metric_bootstrap['Window'] == '13d'].iloc[0]

    r = e13['Rank_Biserial']
    mwu_p = e13['MWU_p']
    bm_p = e13['BM_p']
    perm_p = e13['Perm_p']
    p_ct = e13['P_Ctrl_GT_Trt']

    # Count significant tests at 13d
    n_sig = sum([mwu_p < 0.05, bm_p < 0.05, perm_p < 0.05, b13['TM_Excludes_Zero']])

    print(f"  13-day endpoint (primary):")
    print(f"    Rank-biserial r = {r:+.4f} ({e13['R_Interpret']})")
    print(f"    P(Control > Treatment) = {p_ct:.3f}")
    print(f"    Tests significant (p<0.05): {n_sig}/4")
    print(f"      MWU p={mwu_p:.4f}, BM p={bm_p:.4f}, Perm p={perm_p:.4f}, Bootstrap CI excl 0: {b13['TM_Excludes_Zero']}")

    # Overall pattern across windows
    all_r = metric_effects['Rank_Biserial'].values
    all_p = metric_effects['MWU_p'].values
    consistent_direction = all(r > 0 for r in all_r) or all(r < 0 for r in all_r)
    any_sig = any(p < 0.05 for p in all_p)

    print(f"\n  Pattern across time windows:")
    print(f"    Direction consistent: {'YES' if consistent_direction else 'NO (direction changes)'}")
    print(f"    r values: {', '.join(f'{r:+.3f}' for r in all_r)}")
    print(f"    Any window significant (MWU): {'YES' if any_sig else 'NO'}")

    # Verdict
    if n_sig >= 3 and r > 0:
        verdict = "STRONG evidence of suppression"
    elif n_sig >= 1 and r > 0:
        verdict = "MODERATE evidence of suppression (not all tests agree)"
    elif r > 0 and n_sig == 0:
        verdict = "WEAK/suggestive suppression (direction correct but not significant)"
    elif r < 0 and any_sig:
        verdict = "Evidence AGAINST suppression (Treatment grew MORE)"
    else:
        verdict = "NO clear evidence of suppression"

    print(f"\n  ➤ VERDICT: {verdict}")

print(f"\n{'═' * 75}")
print("END OF SECTION 3.2")
print(f"{'═' * 75}")

## 3.3: Temporal Dynamics — When Does Suppression Emerge?

### Motivation

Sections 3.1 and 3.2 examined the effect at fixed time snapshots (1d, 3d, 7d, 13d). But the *dynamics* of suppression matter just as much as the final magnitude:

- **Does the CN effect appear immediately** (Day 1), or does it take time to manifest?
- **Does the gap between groups widen over time**, or does it plateau?
- **Is there a "critical window"** after which the CN no longer makes a difference?

These questions have direct practical implications. If CNs work within 24 hours, response speed may be less critical than we think. If the effect takes a week to emerge, early monitoring might show "no effect" even when one is developing. If the gap plateaus, there's a natural evaluation timepoint.

### Approach

We analyze the **full day-by-day trajectory** (Day 0 through Day 13), computing at each day:

1. **Effect size trajectory** — rank-biserial r at each day, showing how the treatment effect evolves
2. **Trajectory plot with confidence bands** — median growth curves with bootstrapped 95% CIs, revealing when the groups visually diverge
3. **Day-by-day significance testing** — at which day does the difference first become (and remain) significant?
4. **Difference-in-growth trajectory** — directly plotting Control growth minus Treatment growth over time, with a CI band, to show the *net suppression* trajectory

Together, these paint a complete picture of the suppression dynamics.

### 3.3.1: Day-by-Day Effect Size Trajectory

For each day (0–13), we compute the rank-biserial r comparing Control vs Treatment growth-from-Day-0. This shows how the effect size *evolves* — whether it grows, shrinks, or fluctuates.

We also mark significance (p < 0.05) at each day, allowing us to identify the **suppression onset day** — the first day at which the effect becomes statistically significant.

Note: Day 0 is the baseline (growth = 0 for both groups by definition), so we start from Day 1.

In [ ]:
# --------------------------------------------------
# 3.3.1: DAY-BY-DAY EFFECT SIZE TRAJECTORY
# --------------------------------------------------

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

METRICS = ['Views', 'Likes', 'Shares']

ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

# Compute day-by-day growth from Day 0 and test at each day
daily_effects = []

for metric in METRICS:
    for day in range(1, 14):  # Day 1 through Day 13
        day0_col = f'{metric}_Day0'
        day_col = f'{metric}_Day{day}'

        # Growth from Day 0 to this day
        ctrl_growth = ((ctrl[day_col] - ctrl[day0_col]) / (ctrl[day0_col] + 1) * 100).dropna().values
        trt_growth = ((trt[day_col] - trt[day0_col]) / (trt[day0_col] + 1) * 100).dropna().values

        # Mann-Whitney U + rank-biserial
        mwu_stat, mwu_p = stats.mannwhitneyu(ctrl_growth, trt_growth, alternative='two-sided')
        n1, n2 = len(ctrl_growth), len(trt_growth)
        r = (2 * mwu_stat) / (n1 * n2) - 1

        # P(C>T) from BM ranks
        combined = np.concatenate([ctrl_growth, trt_growth])
        ranks = stats.rankdata(combined)
        p_ct = (np.mean(ranks[:n1]) - (n1 + 1) / 2) / n2

        daily_effects.append({
            'Metric': metric, 'Day': day,
            'r': r, 'p': mwu_p, 'P_CT': p_ct,
            'Ctrl_Median': np.median(ctrl_growth),
            'Trt_Median': np.median(trt_growth),
            'Ctrl_Mean': np.mean(ctrl_growth),
            'Trt_Mean': np.mean(trt_growth),
            'Significant': mwu_p < 0.05
        })

daily_df = pd.DataFrame(daily_effects)

# Print summary table
print("=" * 100)
print("TABLE 8: DAY-BY-DAY EFFECT SIZES (rank-biserial r)")
print("=" * 100)

for metric in METRICS:
    mdf = daily_df[daily_df['Metric'] == metric]
    print(f"\n  {metric.upper()}")
    print(f"  {'Day':<6} {'r':>8} {'p':>10} {'P(C>T)':>8} {'Ctrl Med%':>11} {'Trt Med%':>11} {'Sig':>5}")
    print(f"  {'─' * 62}")

    first_sig = None
    for _, row in mdf.iterrows():
        sig = "*" if row['Significant'] else ""
        if row['Significant'] and first_sig is None:
            first_sig = int(row['Day'])
        print(f"  Day {int(row['Day']):<3} {row['r']:>+7.4f} {row['p']:>10.4f} {row['P_CT']:>8.3f} {row['Ctrl_Median']:>10.1f}% {row['Trt_Median']:>10.1f}% {sig:>5}")

    if first_sig:
        print(f"\n  → Suppression onset: Day {first_sig}")
    else:
        print(f"\n  → No significant suppression detected at any day")

### 3.3.2: Effect Size Evolution Plot

Visualizes the rank-biserial r over time for each metric. This is the key "dynamics" figure:
- The y-axis shows effect size (positive = suppression, negative = anti-suppression)
- Significant days are marked with filled markers
- The dashed line at r = 0 represents no effect

This reveals whether the effect **builds up** gradually, **appears suddenly**, or **fluctuates**.

In [ ]:
# --------------------------------------------------
# 3.3.2: EFFECT SIZE EVOLUTION PLOT
# --------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Effect Size (Rank-Biserial r) Over Time: Does Suppression Build Up?',
             fontsize=14, fontweight='bold', y=1.02)

colors = {'Views': '#4C72B0', 'Likes': '#DD8452', 'Shares': '#55A868'}

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    mdf = daily_df[daily_df['Metric'] == metric]

    days = mdf['Day'].values
    r_vals = mdf['r'].values
    sig = mdf['Significant'].values

    # Plot non-significant days as open markers
    ax.plot(days[~sig], r_vals[~sig], 'o', color=colors[metric],
            markersize=7, markerfacecolor='white', markeredgewidth=2, zorder=5, label='p ≥ 0.05')
    # Plot significant days as filled markers
    ax.plot(days[sig], r_vals[sig], 'o', color=colors[metric],
            markersize=8, zorder=5, label='p < 0.05')
    # Connect with line
    ax.plot(days, r_vals, '-', color=colors[metric], alpha=0.5, linewidth=1.5)

    ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
    ax.set_xlabel('Day')
    ax.set_ylabel('Rank-Biserial r\n(+ve = suppression)')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(range(1, 14))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Add effect size interpretation bands
    ax.axhspan(0.1, 0.3, alpha=0.05, color='green')
    ax.axhspan(-0.3, -0.1, alpha=0.05, color='red')
    ax.set_ylim(min(-0.25, r_vals.min() - 0.05), max(0.25, r_vals.max() + 0.05))

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_3_effect_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3.3: Median Growth Trajectories with Confidence Bands

The effect size plot (3.3.2) shows the *between-group comparison* at each day. This complementary plot shows the *actual growth curves* for each group — how the median tweet in each group accumulates engagement over time.

We add **bootstrapped 95% confidence bands** around each group's median trajectory. Where the bands **don't overlap**, we have visual evidence of divergence. This is more informative than just plotting the medians (as the original notebook did) because it conveys uncertainty.

The bootstrap uses 5,000 resamples at each day — we resample tweets (not days), preserving the within-tweet correlation structure.

In [ ]:
# --------------------------------------------------
# 3.3.3: MEDIAN GROWTH TRAJECTORIES WITH BOOTSTRAP CIs
# --------------------------------------------------

np.random.seed(42)
N_BOOT = 5000

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Median Growth Trajectory with 95% Bootstrap CI',
             fontsize=14, fontweight='bold', y=1.02)

for idx, metric in enumerate(METRICS):
    ax = axes[idx]

    days = range(1, 14)

    for group_name, group_df, color, marker in [
        ('Control', ctrl, '#4C72B0', 'o'),
        ('Treatment', trt, '#DD8452', 's')
    ]:
        medians = []
        ci_lo = []
        ci_hi = []

        day0_col = f'{metric}_Day0'
        day0_vals = group_df[day0_col].values

        for day in days:
            day_col = f'{metric}_Day{day}'
            day_vals = group_df[day_col].values

            # Growth from Day 0
            growth = (day_vals - day0_vals) / (day0_vals + 1) * 100
            medians.append(np.median(growth))

            # Bootstrap CI for median
            boot_medians = []
            n = len(growth)
            for _ in range(N_BOOT):
                boot_idx = np.random.randint(0, n, n)
                boot_medians.append(np.median(growth[boot_idx]))

            ci_lo.append(np.percentile(boot_medians, 2.5))
            ci_hi.append(np.percentile(boot_medians, 97.5))

        ax.plot(list(days), medians, f'-{marker}', color=color, label=group_name,
                markersize=5, linewidth=2, zorder=5)
        ax.fill_between(list(days), ci_lo, ci_hi, alpha=0.15, color=color)

    ax.set_xlabel('Day')
    ax.set_ylabel('Median % Growth from Day 0')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(range(1, 14))
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_3_trajectory_with_ci.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3.4: Net Suppression Trajectory (Difference Plot)

Instead of plotting two curves and eyeballing the gap, this plot directly shows the **difference** between Control and Treatment median growth at each day, with a bootstrapped 95% CI.

This is the clearest visual answer to "is there suppression, and when does it emerge?":
- Values **above zero** = Control grew more = suppression
- Values **below zero** = Treatment grew more = anti-suppression
- CI **excluding zero** = statistically significant at that day

This is particularly useful when the two group trajectories are close together and the gap is hard to see in the trajectory plot.

In [ ]:
# --------------------------------------------------
# 3.3.4: NET SUPPRESSION TRAJECTORY (DIFFERENCE PLOT)
# --------------------------------------------------

np.random.seed(42)
N_BOOT = 5000

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Net Suppression: Median Growth Difference (Control − Treatment) Over Time',
             fontsize=14, fontweight='bold', y=1.02)

for idx, metric in enumerate(METRICS):
    ax = axes[idx]

    days = range(1, 14)
    diffs = []
    ci_lo = []
    ci_hi = []
    sig_days = []

    ctrl_day0 = ctrl[f'{metric}_Day0'].values
    trt_day0 = trt[f'{metric}_Day0'].values

    for day in days:
        ctrl_growth = ((ctrl[f'{metric}_Day{day}'].values - ctrl_day0) / (ctrl_day0 + 1) * 100)
        trt_growth = ((trt[f'{metric}_Day{day}'].values - trt_day0) / (trt_day0 + 1) * 100)

        obs_diff = np.median(ctrl_growth) - np.median(trt_growth)
        diffs.append(obs_diff)

        # Bootstrap CI for the median difference
        boot_diffs = []
        n_c, n_t = len(ctrl_growth), len(trt_growth)
        for _ in range(N_BOOT):
            c_boot = np.median(ctrl_growth[np.random.randint(0, n_c, n_c)])
            t_boot = np.median(trt_growth[np.random.randint(0, n_t, n_t)])
            boot_diffs.append(c_boot - t_boot)

        lo, hi = np.percentile(boot_diffs, [2.5, 97.5])
        ci_lo.append(lo)
        ci_hi.append(hi)

        # Check if CI excludes zero
        if lo > 0 or hi < 0:
            sig_days.append(day)

    days_list = list(days)

    # Plot the difference line
    ax.plot(days_list, diffs, '-o', color='#4C72B0', markersize=6, linewidth=2, zorder=5)
    ax.fill_between(days_list, ci_lo, ci_hi, alpha=0.2, color='#4C72B0')

    # Mark significant days
    for d in sig_days:
        d_idx = d - 1  # 0-indexed
        ax.plot(d, diffs[d_idx], 'D', color='#C44E52', markersize=8, zorder=6)

    ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.7)
    ax.set_xlabel('Day')
    ax.set_ylabel('Median Growth Diff (C − T) in pp')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(range(1, 14))
    ax.grid(True, alpha=0.3)

    # Custom legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='#4C72B0', markersize=6, label='Difference (C−T)'),
        Line2D([0], [0], marker='D', color='#C44E52', linestyle='None', markersize=8, label='CI excludes 0'),
        plt.fill_between([], [], [], alpha=0.2, color='#4C72B0', label='95% Bootstrap CI')
    ]
    ax.legend(handles=legend_elements[:2], fontsize=9, loc='best')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_3_net_suppression.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3.5: Key Takeaways from Section 3.3

In [ ]:
# --------------------------------------------------
# 3.3.5: AUTOMATED KEY TAKEAWAYS
# --------------------------------------------------

print("=" * 75)
print("SECTION 3.3 — KEY TAKEAWAYS")
print("=" * 75)

for metric in METRICS:
    mdf = daily_df[daily_df['Metric'] == metric]
    sig_days = mdf[mdf['Significant']]['Day'].tolist()
    r_vals = mdf['r'].values

    print(f"\n{'─' * 65}")
    print(f"  {metric.upper()}")
    print(f"{'─' * 65}")

    # Direction consistency
    if all(r > 0 for r in r_vals):
        print(f"  Direction: Consistently positive r (suppression direction) across all days.")
    elif all(r < 0 for r in r_vals):
        print(f"  Direction: Consistently negative r (anti-suppression) across all days.")
    else:
        pos_days = sum(1 for r in r_vals if r > 0)
        print(f"  Direction: Mixed — positive r on {pos_days}/13 days, negative on {13-pos_days}/13.")

    # Effect size range
    print(f"  Effect size range: r = {min(r_vals):+.4f} to {max(r_vals):+.4f}")

    # Significance pattern
    if len(sig_days) == 0:
        print(f"  Significance: No days reached p < 0.05.")
        print(f"  → No detectable suppression onset.")
    else:
        print(f"  Significant days: {', '.join(f'Day {int(d)}' for d in sig_days)} ({len(sig_days)}/13 days)")
        print(f"  → First significant day: Day {int(sig_days[0])}")

        # Check if significance is sustained
        consecutive = True
        for i in range(1, len(sig_days)):
            if sig_days[i] != sig_days[i-1] + 1:
                consecutive = False
                break

        if consecutive and len(sig_days) > 1:
            print(f"  → Sustained from Day {int(sig_days[0])} through Day {int(sig_days[-1])}")
        elif len(sig_days) > 1:
            print(f"  → Intermittent significance (not sustained)")

    # Trajectory shape
    early_r = np.mean(r_vals[:3])   # Days 1-3
    mid_r = np.mean(r_vals[3:7])    # Days 4-7
    late_r = np.mean(r_vals[7:])    # Days 8-13

    print(f"\n  Trajectory shape:")
    print(f"    Early (Days 1-3):   mean r = {early_r:+.4f}")
    print(f"    Middle (Days 4-7):  mean r = {mid_r:+.4f}")
    print(f"    Late (Days 8-13):   mean r = {late_r:+.4f}")

    if abs(late_r) > abs(early_r) * 1.5:
        print(f"    → Effect GROWS over time")
    elif abs(late_r) < abs(early_r) * 0.5:
        print(f"    → Effect FADES over time")
    else:
        print(f"    → Effect is relatively STABLE across the monitoring period")

print(f"\n{'═' * 75}")
print("END OF SECTION 3.3")
print(f"{'═' * 75}")

## 3.4: Robustness & Alternative Operationalizations

### Motivation

Sections 3.1–3.3 revealed a nuanced picture:
- **Views**: Treatment tweets gained *more* views — significant anti-suppression at the rank level
- **Likes/Shares**: Directionally consistent with suppression (means, trimmed means) but not significant at the rank level

A natural concern is whether these findings are robust or artifacts of specific analytical choices. This section tests robustness from multiple angles and introduces an important alternative way to measure suppression.

### Key analyses in this section:

**3.4.1 — Baseline-Adjusted Regression**: Controls for Day 0 engagement levels, ensuring that any residual imbalance between groups isn't driving the results.

**3.4.2 — Stratified Analysis by Baseline Engagement**: Tests whether the CN effect differs for low- vs mid- vs high-visibility tweets. If CNs only suppress high-visibility tweets, the aggregate analysis might miss a real effect.

**3.4.3 — Engagement Rate Analysis** ⭐: This is the most important new analysis. Since CNs increase Views (exposure), raw Likes and Shares counts conflate two effects: (1) more people see the tweet, and (2) people who see it may react differently. By analyzing **Likes/Views** and **Shares/Views** *rates*, we isolate the second mechanism — does the CN change how viewers *react* to the harmful content?

**3.4.4 — Outlier Sensitivity**: Re-runs the main tests after removing extreme outliers (top 1% and top 5%) to check whether a handful of viral tweets are driving the results.

### 3.4.1: Baseline-Adjusted Regression

Even though randomization should balance baseline characteristics, our balance check (Step 2) showed that while *medians* were similar at Day 0, *means* differed (e.g., Control Views mean 110 vs Treatment 162). This doesn't invalidate the experiment, but controlling for baseline can:

1. **Remove residual noise** — if some tweets started with higher engagement, their growth trajectory may differ regardless of treatment
2. **Increase statistical power** — by absorbing variance explained by baseline, the treatment coefficient becomes more precise

We use two complementary regression approaches:

**OLS on log-transformed growth**: `log(Day13 + 1) − log(Day0 + 1) ~ Group + log(Day0 + 1)`
- This models *multiplicative* growth, which is natural for engagement data
- The coefficient on Group estimates the treatment effect as a percentage change in the growth multiplier

**Quantile regression at τ = 0.5** (median): Same specification but estimates the treatment effect *at the median*, which is robust to outliers. This addresses the mean vs median divergence we've been seeing.


In [ ]:
# --------------------------------------------------
# 3.4.1: BASELINE-ADJUSTED REGRESSION
# --------------------------------------------------

import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd
from scipy import stats

METRICS = ['Views', 'Likes', 'Shares']

ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

# Prepare regression data
reg_df = complete_df[['Group']].copy()
reg_df['Treatment'] = (reg_df['Group'] == 'Treatment').astype(int)

for metric in METRICS:
    reg_df[f'{metric}_Day0'] = complete_df[f'{metric}_Day0']
    reg_df[f'{metric}_Day13'] = complete_df[f'{metric}_Day13']
    # Log-transformed growth: log(end+1) - log(start+1)
    reg_df[f'{metric}_LogGrowth'] = (
        np.log1p(complete_df[f'{metric}_Day13']) - np.log1p(complete_df[f'{metric}_Day0'])
    )
    reg_df[f'{metric}_LogBase'] = np.log1p(complete_df[f'{metric}_Day0'])
    # Percentage growth (same as before)
    reg_df[f'{metric}_PctGrowth'] = complete_df[f'{metric}_Growth_13d']

print("=" * 90)
print("TABLE 9: BASELINE-ADJUSTED REGRESSION (13-day endpoint)")
print("=" * 90)

for metric in METRICS:
    print(f"\n{'━' * 90}")
    print(f"  {metric.upper()}")
    print(f"{'━' * 90}")

    # --- OLS on log growth ---
    formula = f'{metric}_LogGrowth ~ Treatment + {metric}_LogBase'
    ols_model = smf.ols(formula, data=reg_df).fit()

    coef = ols_model.params['Treatment']
    se = ols_model.bse['Treatment']
    p = ols_model.pvalues['Treatment']
    ci = ols_model.conf_int().loc['Treatment']

    print(f"\n  OLS on log-growth (controls for baseline):")
    print(f"    Treatment coefficient: {coef:+.4f} (SE={se:.4f})")
    print(f"    95% CI: [{ci[0]:+.4f}, {ci[1]:+.4f}]")
    print(f"    p-value: {p:.4f} {'*' if p < 0.05 else ''}")
    print(f"    Interpretation: Treatment {'decreased' if coef < 0 else 'increased'} log-growth by {abs(coef):.4f}")
    print(f"    (≈ {(np.exp(coef) - 1) * 100:+.1f}% change in growth multiplier)")
    print(f"    R² = {ols_model.rsquared:.4f}")

    # --- Quantile regression (median) ---
    formula_qr = f'{metric}_PctGrowth ~ Treatment + {metric}_LogBase'
    try:
        qr_model = smf.quantreg(formula_qr, data=reg_df).fit(q=0.5)

        coef_qr = qr_model.params['Treatment']
        se_qr = qr_model.bse['Treatment']
        p_qr = qr_model.pvalues['Treatment']
        ci_qr = qr_model.conf_int().loc['Treatment']

        print(f"\n  Quantile Regression (median, τ=0.5):")
        print(f"    Treatment coefficient: {coef_qr:+.4f} pp (SE={se_qr:.4f})")
        print(f"    95% CI: [{ci_qr[0]:+.4f}, {ci_qr[1]:+.4f}] pp")
        print(f"    p-value: {p_qr:.4f} {'*' if p_qr < 0.05 else ''}")
        print(f"    Interpretation: Treatment shifted median growth by {coef_qr:+.1f} percentage points")
    except Exception as e:
        print(f"\n  Quantile Regression: Failed ({e})")

### 3.4.2: Stratified Analysis by Baseline Engagement

Does the CN effect depend on how visible the tweet already was when we detected it? This is practically important: if CNs only work on tweets that are already gaining traction, the intervention strategy should prioritize high-visibility tweets. If they work across the board, broad deployment is justified.

We split tweets into **terciles** based on Day 0 Views (Low / Medium / High baseline visibility) and repeat the Mann-Whitney U test within each stratum. This also serves as a robustness check — if the aggregate effect is driven entirely by one stratum, that's important context.


In [ ]:
# --------------------------------------------------
# 3.4.2: STRATIFIED ANALYSIS BY BASELINE ENGAGEMENT
# --------------------------------------------------

print("=" * 100)
print("TABLE 10: STRATIFIED ANALYSIS BY BASELINE VIEWS (Day 0)")
print("=" * 100)

# Create terciles based on Day 0 Views
views_terciles = pd.qcut(complete_df['Views_Day0'], q=3, labels=['Low', 'Medium', 'High'])
complete_df['Views_Tercile'] = views_terciles

print(f"\nTercile boundaries (Views Day 0):")
for label in ['Low', 'Medium', 'High']:
    subset = complete_df[complete_df['Views_Tercile'] == label]['Views_Day0']
    print(f"  {label}: {subset.min():.0f} – {subset.max():.0f} (n={len(subset)})")

print(f"\n{'─' * 100}")

for metric in METRICS:
    print(f"\n  {metric.upper()}")
    print(f"  {'Tercile':<10} {'n_C':>6} {'n_T':>6} │ {'Ctrl Med%':>11} {'Trt Med%':>11} {'r':>8} {'p':>10} {'Sig':>5} │ {'Ctrl Mean%':>12} {'Trt Mean%':>12}")
    print(f"  {'─' * 98}")

    for tercile in ['Low', 'Medium', 'High']:
        mask = complete_df['Views_Tercile'] == tercile
        ctrl_strat = complete_df[mask & (complete_df['Group'] == 'Control')]
        trt_strat = complete_df[mask & (complete_df['Group'] == 'Treatment')]

        col = f'{metric}_Growth_13d'
        ctrl_vals = ctrl_strat[col].dropna().values
        trt_vals = trt_strat[col].dropna().values

        if len(ctrl_vals) >= 10 and len(trt_vals) >= 10:
            mwu_stat, mwu_p = stats.mannwhitneyu(ctrl_vals, trt_vals, alternative='two-sided')
            n1, n2 = len(ctrl_vals), len(trt_vals)
            r = (2 * mwu_stat) / (n1 * n2) - 1
            sig = "*" if mwu_p < 0.05 else ""

            print(f"  {tercile:<10} {n1:>6} {n2:>6} │ {np.median(ctrl_vals):>+10.1f}% {np.median(trt_vals):>+10.1f}% {r:>+7.4f} {mwu_p:>10.4f} {sig:>5} │ {np.mean(ctrl_vals):>+11.1f}% {np.mean(trt_vals):>+11.1f}%")
        else:
            print(f"  {tercile:<10} {len(ctrl_vals):>6} {len(trt_vals):>6} │ {'(insufficient data)':<60}")
    print()

### 3.4.3: Engagement Rate Analysis ⭐

This is a critical analysis motivated by our findings so far:

**The puzzle**: CNs *increase* Views (Treatment tweets get more visibility) but the mean/trimmed mean for Likes and Shares is lower in Treatment. How do we reconcile this?

**The hypothesis**: CNs may suppress the *propensity to engage positively* with harmful content. Even though more people see the tweet (boosted by the CN reply generating activity), a smaller *fraction* of viewers choose to Like or Share it. If true, this is actually a strong form of suppression — the CN doesn't prevent exposure, but it changes how people *react* to the content.

**Operationalization**: We compute **engagement rates**:
- `Like Rate = Likes / (Views + 1)` — the fraction of viewers who liked
- `Share Rate = Shares / (Views + 1)` — the fraction of viewers who shared

We compute these at Day 0 and Day 13, then analyze:
1. The **change in rate** (Day 13 rate − Day 0 rate): did the engagement rate go up or down?
2. The **Day 13 rate** itself: what's the final engagement rate in each group?

If CNs suppress engagement rates, Treatment tweets should show *lower* rates at Day 13 and/or a larger *decline* in rates over time, even though their absolute Views are higher.

Note: The `+1` in the denominator handles tweets with 0 views (same convention as our growth metric).

In [ ]:
# --------------------------------------------------
# 3.4.3: ENGAGEMENT RATE ANALYSIS
# --------------------------------------------------

print("=" * 100)
print("TABLE 11: ENGAGEMENT RATE ANALYSIS (Likes/Views and Shares/Views)")
print("=" * 100)

rate_results = []

for rate_metric in ['Likes', 'Shares']:
    # Compute rates at Day 0 and Day 13
    for day_label, day_num in [('Day0', 0), ('Day13', 13)]:
        complete_df[f'{rate_metric}_Rate_{day_label}'] = (
            complete_df[f'{rate_metric}_{day_label}'] / (complete_df[f'Views_{day_label}'] + 1)
        )

    # Change in rate
    complete_df[f'{rate_metric}_Rate_Change'] = (
        complete_df[f'{rate_metric}_Rate_Day13'] - complete_df[f'{rate_metric}_Rate_Day0']
    )

# Update group splits
ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

for rate_metric in ['Likes', 'Shares']:
    print(f"\n{'━' * 100}")
    print(f"  {rate_metric.upper()} / VIEWS RATE")
    print(f"{'━' * 100}")

    # --- Day 0 rates (should be balanced) ---
    rate_col_0 = f'{rate_metric}_Rate_Day0'
    ctrl_rate0 = ctrl[rate_col_0].dropna().values
    trt_rate0 = trt[rate_col_0].dropna().values
    _, p_balance = stats.mannwhitneyu(ctrl_rate0, trt_rate0, alternative='two-sided')

    print(f"\n  Baseline (Day 0) rate:")
    print(f"    Control: median={np.median(ctrl_rate0):.4f}, mean={np.mean(ctrl_rate0):.4f}")
    print(f"    Treatment: median={np.median(trt_rate0):.4f}, mean={np.mean(trt_rate0):.4f}")
    print(f"    Balance check (MWU p): {p_balance:.4f} {'✓ balanced' if p_balance > 0.05 else '⚠ imbalanced'}")

    # --- Day 13 rates ---
    rate_col_13 = f'{rate_metric}_Rate_Day13'
    ctrl_rate13 = ctrl[rate_col_13].dropna().values
    trt_rate13 = trt[rate_col_13].dropna().values

    mwu_stat, mwu_p = stats.mannwhitneyu(ctrl_rate13, trt_rate13, alternative='two-sided')
    n1, n2 = len(ctrl_rate13), len(trt_rate13)
    r = (2 * mwu_stat) / (n1 * n2) - 1

    # P(C>T) from BM
    combined = np.concatenate([ctrl_rate13, trt_rate13])
    ranks = stats.rankdata(combined)
    p_ct = (np.mean(ranks[:n1]) - (n1 + 1) / 2) / n2

    bm_stat, bm_p = stats.brunnermunzel(ctrl_rate13, trt_rate13)

    print(f"\n  Final (Day 13) rate:")
    print(f"    Control: median={np.median(ctrl_rate13):.4f}, mean={np.mean(ctrl_rate13):.4f}")
    print(f"    Treatment: median={np.median(trt_rate13):.4f}, mean={np.mean(trt_rate13):.4f}")
    print(f"    MWU: r={r:+.4f}, p={mwu_p:.4f} {'*' if mwu_p < 0.05 else ''}")
    print(f"    BM: P(C>T)={p_ct:.3f}, p={bm_p:.4f} {'*' if bm_p < 0.05 else ''}")

    if r > 0:
        print(f"    → Control has HIGHER rate = Treatment viewers engage LESS = SUPPRESSION ✓")
    else:
        print(f"    → Treatment has HIGHER rate = Treatment viewers engage MORE")

    # --- Rate change (Day 13 - Day 0) ---
    rate_change_col = f'{rate_metric}_Rate_Change'
    ctrl_change = ctrl[rate_change_col].dropna().values
    trt_change = trt[rate_change_col].dropna().values

    mwu_stat2, mwu_p2 = stats.mannwhitneyu(ctrl_change, trt_change, alternative='two-sided')
    r2 = (2 * mwu_stat2) / (len(ctrl_change) * len(trt_change)) - 1

    print(f"\n  Rate change (Day 13 − Day 0):")
    print(f"    Control: median={np.median(ctrl_change):.5f}, mean={np.mean(ctrl_change):.5f}")
    print(f"    Treatment: median={np.median(trt_change):.5f}, mean={np.mean(trt_change):.5f}")
    print(f"    MWU: r={r2:+.4f}, p={mwu_p2:.4f} {'*' if mwu_p2 < 0.05 else ''}")

    if r2 > 0:
        print(f"    → Control rate INCREASED more = Treatment rate DROPPED more = SUPPRESSION ✓")
    elif r2 < 0 and np.median(trt_change) < np.median(ctrl_change):
        print(f"    → Nuanced: check mean vs median direction")
    else:
        print(f"    → Treatment rate increased more or dropped less")

    rate_results.append({
        'Metric': rate_metric, 'Day13_r': r, 'Day13_p': mwu_p,
        'Change_r': r2, 'Change_p': mwu_p2, 'P_CT': p_ct
    })

### 3.4.3b: Engagement Rate — Day-by-Day Trajectory

To see *when* engagement rates diverge, we compute the Like Rate and Share Rate at each day and plot the trajectories with bootstrap CIs — the same approach as Section 3.3.3 but now for rates instead of raw counts.


In [ ]:
# --------------------------------------------------
# 3.4.3b: ENGAGEMENT RATE TRAJECTORY
# --------------------------------------------------

np.random.seed(42)
N_BOOT = 5000

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Engagement Rate Trajectory (Metric / Views) Over Time',
             fontsize=14, fontweight='bold', y=1.02)

for idx, rate_metric in enumerate(['Likes', 'Shares']):
    ax = axes[idx]

    days = range(0, 14)

    for group_name, group_df, color, marker in [
        ('Control', ctrl, '#4C72B0', 'o'),
        ('Treatment', trt, '#DD8452', 's')
    ]:
        medians = []
        ci_lo = []
        ci_hi = []

        for day in days:
            views_col = f'Views_Day{day}'
            metric_col = f'{rate_metric}_Day{day}'

            rate = group_df[metric_col].values / (group_df[views_col].values + 1)
            medians.append(np.median(rate))

            # Bootstrap
            boot_medians = []
            n = len(rate)
            for _ in range(N_BOOT):
                boot_idx = np.random.randint(0, n, n)
                boot_medians.append(np.median(rate[boot_idx]))
            ci_lo.append(np.percentile(boot_medians, 2.5))
            ci_hi.append(np.percentile(boot_medians, 97.5))

        ax.plot(list(days), medians, f'-{marker}', color=color, label=group_name,
                markersize=5, linewidth=2, zorder=5)
        ax.fill_between(list(days), ci_lo, ci_hi, alpha=0.15, color=color)

    ax.set_xlabel('Day')
    ax.set_ylabel(f'Median {rate_metric}/Views Rate')
    ax.set_title(f'{rate_metric} per View', fontsize=13, fontweight='bold')
    ax.set_xticks(range(0, 14))
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_4_rate_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4.4: Outlier Sensitivity Analysis

Section 3.1 showed extreme skewness and significant outliers, especially for Views (Control mean 6911% vs median 125%). Here we re-run the core Mann-Whitney test after progressively removing outliers:

1. **Remove top 1%** of growth values (the most extreme outliers)
2. **Remove top 5%** (aggressive trimming)
3. **Winsorize at 95th percentile** (cap extreme values rather than removing them)

If the results change substantially, outliers were driving the effect. If they're stable, the effect is robust.

In [ ]:
# --------------------------------------------------
# 3.4.4: OUTLIER SENSITIVITY ANALYSIS
# --------------------------------------------------

print("=" * 100)
print("TABLE 12: OUTLIER SENSITIVITY — 13-DAY GROWTH")
print("=" * 100)
print("  Re-running Mann-Whitney U after different outlier treatments\n")

for metric in METRICS:
    col = f'{metric}_Growth_13d'
    ctrl_vals = ctrl[col].dropna()
    trt_vals = trt[col].dropna()
    all_vals = pd.concat([ctrl_vals, trt_vals])

    print(f"  {metric.upper()}")
    print(f"  {'Treatment':<25} {'n_C':>6} {'n_T':>6} {'Ctrl Med%':>11} {'Trt Med%':>11} {'r':>8} {'p':>10} {'Sig':>5}")
    print(f"  {'─' * 85}")

    treatments = [
        ('No trimming (original)', ctrl_vals.values, trt_vals.values),
    ]

    # Top 1% removed
    p99 = all_vals.quantile(0.99)
    c1 = ctrl_vals[ctrl_vals <= p99].values
    t1 = trt_vals[trt_vals <= p99].values
    treatments.append((f'Remove top 1% (>{p99:.0f}%)', c1, t1))

    # Top 5% removed
    p95 = all_vals.quantile(0.95)
    c5 = ctrl_vals[ctrl_vals <= p95].values
    t5 = trt_vals[trt_vals <= p95].values
    treatments.append((f'Remove top 5% (>{p95:.0f}%)', c5, t5))

    # Winsorized at 95th percentile
    c_wins = ctrl_vals.clip(upper=p95).values
    t_wins = trt_vals.clip(upper=p95).values
    treatments.append(('Winsorize at 95th pctl', c_wins, t_wins))

    # Bottom & top 5% removed (symmetric)
    p5 = all_vals.quantile(0.05)
    c_sym = ctrl_vals[(ctrl_vals >= p5) & (ctrl_vals <= p95)].values
    t_sym = trt_vals[(trt_vals >= p5) & (trt_vals <= p95)].values
    treatments.append(('Remove top & bottom 5%', c_sym, t_sym))

    for label, c_data, t_data in treatments:
        if len(c_data) >= 10 and len(t_data) >= 10:
            stat, p = stats.mannwhitneyu(c_data, t_data, alternative='two-sided')
            n1, n2 = len(c_data), len(t_data)
            r = (2 * stat) / (n1 * n2) - 1
            sig = "*" if p < 0.05 else ""
            print(f"  {label:<25} {n1:>6} {n2:>6} {np.median(c_data):>+10.1f}% {np.median(t_data):>+10.1f}% {r:>+7.4f} {p:>10.4f} {sig:>5}")

    print()

### 3.4.5: Key Takeaways from Section 3.4

In [ ]:
# --------------------------------------------------
# 3.4.5: AUTOMATED KEY TAKEAWAYS
# --------------------------------------------------

print("=" * 75)
print("SECTION 3.4 — KEY TAKEAWAYS")
print("=" * 75)

print(f"\n{'─' * 65}")
print(f"  BASELINE-ADJUSTED REGRESSION")
print(f"{'─' * 65}")
print(f"  Results above show whether treatment effects hold after")
print(f"  controlling for Day 0 engagement. Check the coefficient")
print(f"  significance and direction for each metric.")

print(f"\n{'─' * 65}")
print(f"  STRATIFIED ANALYSIS")
print(f"{'─' * 65}")
print(f"  Check whether the effect is concentrated in one engagement")
print(f"  tercile or is distributed across tweet sizes.")

print(f"\n{'─' * 65}")
print(f"  ENGAGEMENT RATE ANALYSIS")
print(f"{'─' * 65}")

for res in rate_results:
    metric = res['Metric']
    print(f"\n  {metric}/Views rate:")
    print(f"    Day 13 rate: r={res['Day13_r']:+.4f}, p={res['Day13_p']:.4f}")
    print(f"    Rate change: r={res['Change_r']:+.4f}, p={res['Change_p']:.4f}")

    if res['Day13_r'] > 0 and res['Day13_p'] < 0.05:
        print(f"    → SIGNIFICANT: Viewers of Treatment tweets {metric.lower()[:-1]} LESS per view ✓")
    elif res['Day13_r'] > 0:
        print(f"    → Directional: Treatment viewers {metric.lower()[:-1]} less per view, but not significant")
    else:
        print(f"    → No evidence of rate suppression")

print(f"\n{'─' * 65}")
print(f"  OUTLIER SENSITIVITY")
print(f"{'─' * 65}")
print(f"  Check whether significance and effect direction are stable")
print(f"  across all outlier treatments. Consistent results = robust.")

print(f"\n{'═' * 75}")
print("END OF SECTION 3.4")
print(f"{'═' * 75}")

## 3.5: Individual-Level Distributional Analysis

### Motivation

The preceding sections have been comparing **summary statistics** (medians, means, effect sizes) between groups. But summary statistics can mask important patterns in the full distribution. This is especially true for our data, which is heavily **zero-inflated** — the majority of tweets gain 0 Likes and 0 Shares, making the median uninformative for these metrics.

This section takes a different approach: instead of comparing summaries, we compare the **entire distributions** of engagement growth. This lets us answer questions that summary statistics cannot:

1. **Is Treatment stochastically dominated?** — Is the Treatment growth distribution shifted left *across the board*, or only in certain parts?
2. **Where in the distribution does the effect operate?** — Does the CN suppress growth uniformly, or does it mainly prevent the high-growth tail (viral prevention)?
3. **What fraction of tweets are "suppressed"?** — An intuitive, communicable metric: "X% of treated tweets grew less than the typical control tweet."

### The zero-inflation problem

A core challenge throughout our analysis has been that most tweets gain 0 Likes and 0 Shares. This means:
- Medians are exactly 0 for both groups → median-based comparisons are uninformative
- Mann-Whitney U is essentially comparing "how often is Control's zero vs Treatment's zero" → low power
- The signal, if it exists, is in the **non-zero tail** — the tweets that *did* gain engagement

This section addresses this directly by (a) examining the full CDF (which is informative even with zero-inflation), (b) analyzing the non-zero subset separately, and (c) using a two-part model that handles zeros and non-zeros explicitly.

### 3.5.1: Cumulative Distribution Function (CDF) Comparison

The CDF shows, for any value x, what fraction of tweets had growth ≤ x. If Treatment's CDF is to the **left** of Control's (higher curve at each x), Treatment tweets tend to have lower growth — i.e., suppression.

CDFs are especially powerful for our zero-inflated data because:
- The **vertical gap at x = 0** directly shows the difference in zero-inflation rates (the fraction of tweets with exactly 0 growth)
- The **shape of the upper tail** shows whether high-growth tweets are more or less common in Treatment
- Visual divergence between the curves is the same thing the Mann-Whitney U test is testing — so the CDF plot is literally a visual representation of the test

We also run a **Kolmogorov-Smirnov (KS) test**, which tests whether the two full distributions are different (not just shifted). Unlike Mann-Whitney, KS is sensitive to *any* distributional difference — shape, spread, or location.


In [ ]:
# --------------------------------------------------
# 3.5.1: CDF COMPARISON
# --------------------------------------------------

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

METRICS = ['Views', 'Likes', 'Shares']

ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Cumulative Distribution of 13-Day Engagement Growth: Control vs Treatment',
             fontsize=14, fontweight='bold', y=1.02)

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    col = f'{metric}_Growth_13d'

    ctrl_vals = np.sort(ctrl[col].dropna().values)
    trt_vals = np.sort(trt[col].dropna().values)

    # Clip for visualization (keep full data for tests)
    clip_max = np.percentile(np.concatenate([ctrl_vals, trt_vals]), 95)
    clip_min = np.percentile(np.concatenate([ctrl_vals, trt_vals]), 2)

    # Plot CDFs
    ctrl_cdf_y = np.arange(1, len(ctrl_vals) + 1) / len(ctrl_vals)
    trt_cdf_y = np.arange(1, len(trt_vals) + 1) / len(trt_vals)

    ax.step(ctrl_vals, ctrl_cdf_y, where='post', color='#4C72B0', linewidth=2, label='Control')
    ax.step(trt_vals, trt_cdf_y, where='post', color='#DD8452', linewidth=2, label='Treatment')

    # Mark zero point
    ctrl_zero_frac = np.mean(ctrl_vals <= 0)
    trt_zero_frac = np.mean(trt_vals <= 0)
    ax.axhline(ctrl_zero_frac, color='#4C72B0', linestyle=':', alpha=0.5)
    ax.axhline(trt_zero_frac, color='#DD8452', linestyle=':', alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.3)

    # KS test
    ks_stat, ks_p = stats.ks_2samp(ctrl_vals, trt_vals)

    ax.set_xlim(clip_min, clip_max)
    ax.set_xlabel('Growth %')
    ax.set_ylabel('Cumulative Probability')
    ax.set_title(f'{metric}\nKS stat={ks_stat:.3f}, p={ks_p:.4f}{"*" if ks_p < 0.05 else ""}',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # Annotate zero fractions
    ax.annotate(f'C: {ctrl_zero_frac:.1%} ≤ 0', xy=(0, ctrl_zero_frac), fontsize=8,
                xytext=(clip_max*0.3, ctrl_zero_frac + 0.03), color='#4C72B0')
    ax.annotate(f'T: {trt_zero_frac:.1%} ≤ 0', xy=(0, trt_zero_frac), fontsize=8,
                xytext=(clip_max*0.3, trt_zero_frac - 0.05), color='#DD8452')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_5_cdf.png', dpi=150, bbox_inches='tight')
plt.show()

# Print KS test results
print("\nKolmogorov-Smirnov Test (full distribution comparison):")
for metric in METRICS:
    col = f'{metric}_Growth_13d'
    ks_stat, ks_p = stats.ks_2samp(ctrl[col].dropna(), trt[col].dropna())
    print(f"  {metric}: KS={ks_stat:.4f}, p={ks_p:.4f} {'*' if ks_p < 0.05 else ''}")

### 3.5.2: "Proportion Suppressed" Analysis

An intuitive metric: **what fraction of Treatment tweets grew less than the Control median?**

If the CN has no effect, this should be ~50% (half of any group falls below the other group's median by chance). If CNs suppress growth, significantly more than 50% of Treatment tweets should fall below the Control median.

We test this with a **binomial test** (exact test of whether the observed proportion differs from 0.5).

This metric is also highly communicable: "62% of CN-treated tweets had lower growth than the typical untreated tweet" is a sentence a policymaker can understand.

**Important**: For zero-inflated metrics where the Control median is 0, this test becomes: "what fraction of Treatment tweets had growth ≤ 0?" — which is still informative (it tells us whether Treatment tweets are more or less likely to gain *any* engagement).


In [ ]:
# --------------------------------------------------
# 3.5.2: PROPORTION SUPPRESSED
# --------------------------------------------------

print("=" * 85)
print("TABLE 13: PROPORTION SUPPRESSED")
print("=" * 85)
print("  Fraction of Treatment tweets with growth ≤ Control median\n")

for metric in METRICS:
    col = f'{metric}_Growth_13d'
    ctrl_median = ctrl[col].median()

    trt_vals = trt[col].dropna()
    n_below = (trt_vals <= ctrl_median).sum()
    n_total = len(trt_vals)
    prop = n_below / n_total

    # Binomial test: is proportion significantly different from 0.5?
    binom_p = stats.binomtest(n_below, n_total, 0.5).pvalue

    print(f"  {metric}:")
    print(f"    Control median growth: {ctrl_median:.1f}%")
    print(f"    Treatment tweets ≤ Control median: {n_below}/{n_total} = {prop:.1%}")
    print(f"    Binomial test vs 50%: p = {binom_p:.4f} {'*' if binom_p < 0.05 else ''}")

    if prop > 0.5 and binom_p < 0.05:
        print(f"    → SIGNIFICANT: More Treatment tweets suppressed than expected")
    elif prop > 0.5:
        print(f"    → Directional suppression but not significant")
    elif prop < 0.5 and binom_p < 0.05:
        print(f"    → SIGNIFICANT: Fewer Treatment tweets suppressed = anti-suppression")
    else:
        print(f"    → No clear suppression pattern")
    print()

### 3.5.3: Two-Part Analysis (Zero vs Non-Zero)

For Likes and Shares, where the majority of tweets have 0 growth, a standard comparison is underpowered because it lumps together two fundamentally different types of tweets:
1. Tweets that gained **zero** additional engagement (the majority)
2. Tweets that gained **some** engagement (the interesting minority)

The CN could affect either or both:
- **Part 1 (Extensive margin)**: Does the CN change the *probability* of gaining any engagement? (Chi-square test)
- **Part 2 (Intensive margin)**: Among tweets that *did* gain engagement, does the CN affect *how much* they gained? (Mann-Whitney U on the non-zero subset)

This "two-part model" or "hurdle model" approach is standard in health economics and count data analysis. It avoids the zero-inflation problem by modeling the zeros and non-zeros separately.

**Why this matters**: If the CN doesn't change the probability of gaining a Like (Part 1) but reduces *how many* Likes are gained among those that do get Liked (Part 2), that's a real suppression effect that the aggregate analysis would miss.


In [ ]:
# --------------------------------------------------
# 3.5.3: TWO-PART ANALYSIS
# --------------------------------------------------

print("=" * 100)
print("TABLE 14: TWO-PART ANALYSIS (Zero vs Non-Zero, 13-day growth)")
print("=" * 100)

for metric in METRICS:
    col = f'{metric}_Growth_13d'

    ctrl_vals = ctrl[col].dropna()
    trt_vals = trt[col].dropna()

    # Part 1: Probability of gaining any engagement (growth > 0)
    ctrl_any = (ctrl_vals > 0).sum()
    ctrl_none = (ctrl_vals <= 0).sum()
    trt_any = (trt_vals > 0).sum()
    trt_none = (trt_vals <= 0).sum()

    ctrl_prop = ctrl_any / len(ctrl_vals)
    trt_prop = trt_any / len(trt_vals)

    # Chi-square test for proportions
    contingency = np.array([[ctrl_any, ctrl_none], [trt_any, trt_none]])
    chi2, chi_p, _, _ = stats.chi2_contingency(contingency)

    print(f"\n{'━' * 100}")
    print(f"  {metric.upper()}")
    print(f"{'━' * 100}")

    print(f"\n  PART 1 — Extensive Margin: P(gained any {metric.lower()})")
    print(f"    Control: {ctrl_any}/{len(ctrl_vals)} = {ctrl_prop:.1%}")
    print(f"    Treatment: {trt_any}/{len(trt_vals)} = {trt_prop:.1%}")
    print(f"    Difference: {(ctrl_prop - trt_prop)*100:+.1f} pp")
    print(f"    Chi-square test: χ²={chi2:.3f}, p={chi_p:.4f} {'*' if chi_p < 0.05 else ''}")

    if ctrl_prop > trt_prop:
        print(f"    → Treatment LESS likely to gain {metric.lower()} = SUPPRESSION direction")
    else:
        print(f"    → Treatment MORE likely to gain {metric.lower()}")

    # Part 2: Among non-zero, how much?
    ctrl_nonzero = ctrl_vals[ctrl_vals > 0]
    trt_nonzero = trt_vals[trt_vals > 0]

    print(f"\n  PART 2 — Intensive Margin: Growth among tweets that gained {metric.lower()}")
    print(f"    Control: n={len(ctrl_nonzero)}, median={ctrl_nonzero.median():.1f}%, mean={ctrl_nonzero.mean():.1f}%")
    print(f"    Treatment: n={len(trt_nonzero)}, median={trt_nonzero.median():.1f}%, mean={trt_nonzero.mean():.1f}%")

    if len(ctrl_nonzero) >= 10 and len(trt_nonzero) >= 10:
        mwu_stat, mwu_p = stats.mannwhitneyu(ctrl_nonzero, trt_nonzero, alternative='two-sided')
        n1, n2 = len(ctrl_nonzero), len(trt_nonzero)
        r = (2 * mwu_stat) / (n1 * n2) - 1

        print(f"    MWU: r={r:+.4f}, p={mwu_p:.4f} {'*' if mwu_p < 0.05 else ''}")

        if r > 0:
            print(f"    → Among gainers, Control grew MORE = SUPPRESSION direction ✓")
        else:
            print(f"    → Among gainers, Treatment grew MORE")

        # Also test with trimmed mean (bootstrap)
        np.random.seed(42)
        boot_diffs = []
        for _ in range(5000):
            c_boot = np.random.choice(ctrl_nonzero.values, len(ctrl_nonzero), replace=True)
            t_boot = np.random.choice(trt_nonzero.values, len(trt_nonzero), replace=True)
            boot_diffs.append(stats.trim_mean(c_boot, 0.05) - stats.trim_mean(t_boot, 0.05))

        ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
        tm_diff = stats.trim_mean(ctrl_nonzero.values, 0.05) - stats.trim_mean(trt_nonzero.values, 0.05)
        print(f"    Trimmed mean diff (C−T): {tm_diff:+.1f}pp, 95% CI: [{ci_lo:+.1f}, {ci_hi:+.1f}]")
    else:
        print(f"    (Insufficient non-zero observations for testing)")

### 3.5.4: Quantile-Quantile (Q-Q) Plot

The Q-Q plot compares the quantiles of two distributions directly. Each point represents a matched quantile: the x-coordinate is the Control value at that quantile, the y-coordinate is the Treatment value.

- If the points fall on the **45° diagonal** (y = x), the distributions are identical
- If points are **below the diagonal**, Treatment values are lower at that quantile = suppression
- If points are **above the diagonal**, Treatment values are higher = anti-suppression

This is particularly powerful because it shows **where in the distribution** the effect operates:
- Divergence only in the upper quantiles → CNs prevent viral breakouts
- Divergence only in the lower quantiles → CNs affect typical tweets but not viral ones
- Uniform shift → CNs suppress growth across the board

For Likes and Shares, the lower quantiles will all be at (0, 0) due to zero-inflation, but the upper quantiles are where the action is.


In [ ]:
# --------------------------------------------------
# 3.5.4: Q-Q PLOT
# --------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Q-Q Plot: Treatment Quantiles vs Control Quantiles (13-Day Growth)',
             fontsize=14, fontweight='bold', y=1.02)

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    col = f'{metric}_Growth_13d'

    ctrl_vals = ctrl[col].dropna().values
    trt_vals = trt[col].dropna().values

    # Compute matched quantiles
    quantiles = np.linspace(0, 1, 101)  # 0th to 100th percentile
    ctrl_q = np.percentile(ctrl_vals, quantiles * 100)
    trt_q = np.percentile(trt_vals, quantiles * 100)

    # Clip for visualization
    clip_val = np.percentile(np.concatenate([ctrl_vals, trt_vals]), 97)
    ctrl_q_clipped = np.clip(ctrl_q, -50, clip_val)
    trt_q_clipped = np.clip(trt_q, -50, clip_val)

    # Color points by quantile
    colors = plt.cm.viridis(quantiles)
    ax.scatter(ctrl_q_clipped, trt_q_clipped, c=quantiles, cmap='viridis',
               s=20, alpha=0.8, zorder=5)

    # 45-degree line
    lims = [min(ctrl_q_clipped.min(), trt_q_clipped.min()),
            max(ctrl_q_clipped.max(), trt_q_clipped.max())]
    ax.plot(lims, lims, 'k--', alpha=0.5, linewidth=1, label='y = x (no effect)')

    ax.set_xlabel('Control Quantile (%)')
    ax.set_ylabel('Treatment Quantile (%)')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(0, 100))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.8)
    cbar.set_label('Percentile', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_3_5_qq.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5.5: Key Takeaways from Section 3.5

In [ ]:
# --------------------------------------------------
# 3.5.5: AUTOMATED KEY TAKEAWAYS
# --------------------------------------------------

print("=" * 75)
print("SECTION 3.5 — KEY TAKEAWAYS")
print("=" * 75)

for metric in METRICS:
    col = f'{metric}_Growth_13d'

    ctrl_vals = ctrl[col].dropna()
    trt_vals = trt[col].dropna()

    print(f"\n{'─' * 65}")
    print(f"  {metric.upper()}")
    print(f"{'─' * 65}")

    # KS test
    ks_stat, ks_p = stats.ks_2samp(ctrl_vals, trt_vals)
    print(f"  Distribution test (KS): stat={ks_stat:.4f}, p={ks_p:.4f} {'*' if ks_p < 0.05 else ''}")

    # Proportion suppressed
    ctrl_median = ctrl_vals.median()
    prop = (trt_vals <= ctrl_median).mean()
    binom_p = stats.binomtest(int((trt_vals <= ctrl_median).sum()), len(trt_vals), 0.5).pvalue
    print(f"  Proportion suppressed: {prop:.1%} (binom p={binom_p:.4f})")

    # Two-part summary
    ctrl_prop_any = (ctrl_vals > 0).mean()
    trt_prop_any = (trt_vals > 0).mean()

    ctrl_nz = ctrl_vals[ctrl_vals > 0]
    trt_nz = trt_vals[trt_vals > 0]

    print(f"  Two-part model:")
    print(f"    Part 1 (any gain?): Control {ctrl_prop_any:.1%} vs Treatment {trt_prop_any:.1%}")

    if len(ctrl_nz) >= 10 and len(trt_nz) >= 10:
        mwu_stat, mwu_p = stats.mannwhitneyu(ctrl_nz, trt_nz, alternative='two-sided')
        r = (2 * mwu_stat) / (len(ctrl_nz) * len(trt_nz)) - 1
        print(f"    Part 2 (among gainers): r={r:+.4f}, p={mwu_p:.4f} {'*' if mwu_p < 0.05 else ''}")
        print(f"      Control gainers median: {ctrl_nz.median():.1f}%, Treatment gainers median: {trt_nz.median():.1f}%")

    # Overall interpretation
    signals = []
    if ks_p < 0.05: signals.append("KS")
    if prop > 0.5 and binom_p < 0.05: signals.append("proportion")
    if len(ctrl_nz) >= 10 and len(trt_nz) >= 10 and r > 0 and mwu_p < 0.05: signals.append("intensive margin")
    if ctrl_prop_any > trt_prop_any: signals.append("extensive margin (directional)")

    if len(signals) >= 2:
        print(f"  ➤ CONVERGING evidence from: {', '.join(signals)}")
    elif len(signals) == 1:
        print(f"  ➤ Partial evidence from: {signals[0]}")
    else:
        print(f"  ➤ No significant distributional evidence of suppression")

print(f"\n{'═' * 75}")
print("END OF SECTION 3.5")
print(f"{'═' * 75}")

## Summary of Main Effect Evidence by Metric

The tables below compile **every analysis** from Sections 3.1–3.5, organized by metric. Each row is a specific test or comparison, with the numeric result and whether it supports or contradicts the suppression hypothesis. This is designed to be the definitive reference for interpreting the main effect.

**Reading guide:**
- ✅ = Evidence **for** suppression (Treatment grew less / engaged less)
- ❌ = Evidence **against** suppression (Treatment grew more / engaged more)  
- ➖ = Inconclusive / negligible / not significant in either direction
- **Bold p-values** indicate statistical significance (p < 0.05)
- "C" = Control, "T" = Treatment

---

### 📊 VIEWS

**Overall verdict: Significant ANTI-suppression.** Treatment tweets consistently gained more views than Control tweets. This is expected — the CN reply itself generates activity that boosts the tweet's algorithmic visibility.

| # | Analysis | Result | Direction | Sig? |
|---|----------|--------|-----------|------|
| 1 | **Median growth (13d)** | C: 125.0% vs T: 150.0% | T grew more | ❌ |
| 2 | **Mean growth (13d)** | C: 6911.1% vs T: 502.0% | C grew more (outlier-driven) | ✅ |
| 3 | **Trimmed mean (5%, 13d)** | C: 244.7% vs T: 272.7% | T grew more | ❌ |
| 4 | **Mann-Whitney U** | r = −0.123, **p = 0.0002** | T stochastically larger | ❌ |
| 5 | **Brunner-Munzel** | P(C>T) = 0.438, **p = 0.0002** | T larger 56.2% of the time | ❌ |
| 6 | **Permutation test (median diff)** | **p = 0.0002** | T median higher | ❌ |
| 7 | **Bootstrap CI (trimmed mean diff)** | CI includes 0 | Inconclusive | ➖ |
| 8 | **KS test (full distribution)** | KS = 0.125, **p = 0.0001** | Distributions differ | ❌ |
| 9 | **Proportion suppressed** | 42.4% of T ≤ C median, **p = 0.0002** | Fewer T tweets suppressed | ❌ |
| 10 | **Two-part: Extensive margin** | C: 94.3% vs T: 99.3%, **p < 0.0001** | T more likely to gain views | ❌ |
| 11 | **Two-part: Intensive margin** | C med: 137.5% vs T med: 150.0%, r = −0.079, **p = 0.020** | Among gainers, T grew more | ❌ |
| 12 | **OLS regression (baseline-adjusted)** | Coef = +0.141 (+15.2%), **p = 0.030** | T grew more after controlling for baseline | ❌ |
| 13 | **Quantile regression (median)** | Coef = +25.0 pp, **p = 0.032** | T median higher after controlling for baseline | ❌ |
| 14 | **Temporal pattern** | r consistently negative, significant Day 1–13 | Stable anti-suppression from Day 1 | ❌ |
| 15 | **Outlier sensitivity** | Effect remains significant after all trimming | Robust | ❌ |
| 16 | **Stratified (Low baseline)** | C med: 150% vs T med: 200%, r = −0.155, **p = 0.004** | T grew more in low-visibility tweets | ❌ |
| 17 | **Stratified (Medium baseline)** | C med: 79% vs T med: 121%, r = −0.149, **p = 0.016** | T grew more in mid-visibility tweets | ❌ |
| 18 | **Stratified (High baseline)** | C med: 136% vs T med: 129%, r = −0.053, p = 0.359 | Slight T advantage, not sig | ➖ |

**Summary:** 14 ❌, 1 ✅ (outlier-driven mean only), 2 ➖. The CN reply **boosts tweet visibility** — a clear, robust, and expected side-effect of engaging with the tweet.

---

### 👍 LIKES

**Overall verdict: Directional suppression, not statistically significant.** Every analysis points toward Treatment tweets gaining fewer Likes, but the effect is small and the data is heavily zero-inflated (62–64% of tweets gain zero Likes), limiting statistical power.

| # | Analysis | Result | Direction | Sig? |
|---|----------|--------|-----------|------|
| 1 | **Median growth (13d)** | C: 0.0% vs T: 0.0% | Both zero (uninformative) | ➖ |
| 2 | **Mean growth (13d)** | C: 325.4% vs T: 176.7% | C grew more | ✅ |
| 3 | **Trimmed mean (5%, 13d)** | C: 69.6% vs T: 60.1% | C grew more | ✅ |
| 4 | **Mann-Whitney U** | r = +0.019, p = 0.512 | Slight suppression, not sig | ➖ |
| 5 | **Brunner-Munzel** | P(C>T) = 0.510, p = 0.512 | Negligible | ➖ |
| 6 | **Permutation test (median diff)** | **p = 0.0004** | Median diff significant | ✅ |
| 7 | **Bootstrap CI (trimmed mean diff)** | CI includes 0 | Inconclusive | ➖ |
| 8 | **KS test (full distribution)** | KS = 0.038, p = 0.745 | Distributions indistinguishable | ➖ |
| 9 | **Two-part: Extensive margin** | C: 37.9% vs T: 36.1%, p = 0.556 | T slightly less likely to gain likes | ✅ᵈ |
| 10 | **Two-part: Intensive margin** | C med: 200% vs T med: 108%, r = +0.053, p = 0.324 | Among gainers, C grew more | ✅ᵈ |
| 11 | **Two-part: Intensive (mean)** | C mean: 861% vs T mean: 491% | C gained substantially more | ✅ |
| 12 | **OLS regression (baseline-adjusted)** | Coef = −0.039 (−3.8%), p = 0.396 | T grew less, not sig | ✅ᵈ |
| 13 | **Quantile regression (median)** | Coef = 0.0 pp, p = 1.000 | No effect at median (zero-inflated) | ➖ |
| 14 | **Engagement rate (Day 13)** | C mean: 0.026 vs T mean: 0.023, r = +0.003, p = 0.925 | T viewers like less per view | ✅ᵈ |
| 15 | **Engagement rate change** | C mean Δ: +0.007 vs T mean Δ: +0.003, r = +0.040, p = 0.197 | T rate increased less | ✅ᵈ |
| 16 | **Temporal pattern** | r consistently positive (+0.02 to +0.04), never significant | Stable but weak | ✅ᵈ |
| 17 | **Outlier sensitivity** | Direction stable across all treatments, never significant | Consistent but underpowered | ✅ᵈ |
| 18 | **Stratified (Low baseline)** | C mean: 300% vs T mean: 67%, r = +0.014, p = 0.736 | Suppression in means | ✅ᵈ |
| 19 | **Stratified (High baseline)** | C med: 80% vs T med: 50%, r = +0.031, p = 0.576 | Suppression direction | ✅ᵈ |

ᵈ = directional (correct direction but not statistically significant)

**Summary:** 3 ✅ (significant), 10 ✅ᵈ (directional), 0 ❌, 6 ➖. The suppression signal is **remarkably consistent in direction** — virtually every analysis points toward Treatment tweets gaining fewer Likes — but the effect size is small (r ≈ 0.02) and statistical significance is only achieved in mean-based comparisons and the permutation test. Zero-inflation severely limits power for rank-based tests.

---

### 🔄 SHARES

**Overall verdict: No detectable effect.** Direction is inconsistent and effect sizes are negligible. The data is extremely zero-inflated (84–85% of tweets gain zero Shares), leaving very few observations for meaningful comparison.

| # | Analysis | Result | Direction | Sig? |
|---|----------|--------|-----------|------|
| 1 | **Median growth (13d)** | C: 0.0% vs T: 0.0% | Both zero (uninformative) | ➖ |
| 2 | **Mean growth (13d)** | C: 98.7% vs T: 84.3% | C grew more | ✅ᵈ |
| 3 | **Trimmed mean (5%, 13d)** | C: 13.2% vs T: 12.0% | C grew more | ✅ᵈ |
| 4 | **Mann-Whitney U** | r = +0.002, p = 0.929 | Negligible | ➖ |
| 5 | **Brunner-Munzel** | P(C>T) = 0.501, p = 0.928 | Negligible | ➖ |
| 6 | **Permutation test (median diff)** | **p = 0.0004** | Median diff significant | ✅ |
| 7 | **Bootstrap CI (trimmed mean diff)** | CI includes 0 | Inconclusive | ➖ |
| 8 | **KS test (full distribution)** | KS = 0.011, p = 1.000 | Distributions identical | ➖ |
| 9 | **Two-part: Extensive margin** | C: 15.8% vs T: 15.1%, p = 0.784 | Nearly identical | ➖ |
| 10 | **Two-part: Intensive margin** | C med: 100% vs T med: 100%, r ≈ 0, p = 1.000 | No difference among gainers | ➖ |
| 11 | **Two-part: Intensive (mean)** | C mean: 627% vs T mean: 559% | Slight C advantage | ✅ᵈ |
| 12 | **OLS regression (baseline-adjusted)** | Coef = −0.013 (−1.3%), p = 0.647 | Negligible | ➖ |
| 13 | **Quantile regression (median)** | Coef = 0.0 pp, p = NaN | Zero-inflated, unestimable | ➖ |
| 14 | **Engagement rate (Day 13)** | r = +0.009, p = 0.706 | Negligible | ➖ |
| 15 | **Engagement rate change** | C mean Δ: +0.0012 vs T mean Δ: +0.0004, r = +0.025, p = 0.271 | Slight suppression | ✅ᵈ |
| 16 | **Temporal pattern** | r mixed (12/13 days positive, 1 negative), never significant | No pattern | ➖ |
| 17 | **Outlier sensitivity** | Direction stable, never significant | Consistent but negligible | ➖ |

ᵈ = directional (correct direction but not statistically significant)

**Summary:** 1 ✅, 4 ✅ᵈ, 0 ❌, 12 ➖. The evidence is overwhelmingly inconclusive. With 85% of tweets at zero growth, there is simply insufficient variation to detect an effect, if one exists.

---

### 🔑 Cross-Metric Synthesis

| Aspect | Views | Likes | Shares |
|--------|-------|-------|--------|
| **Direction** | Anti-suppression (more views) | Suppression (fewer likes) | Inconclusive |
| **Significance** | Strong (p < 0.001) | Not significant (suggestive) | Not significant |
| **Effect size** | Small (r ≈ 0.12) | Negligible (r ≈ 0.02) | Negligible (r ≈ 0.00) |
| **Consistency** | 14/17 analyses agree | 13/19 analyses in supp. direction | 5/17 in supp. direction |
| **Robust to outliers?** | Yes | Yes (direction) | N/A |
| **Engagement rate** | N/A (this IS the exposure) | Lower in T (not sig) | Lower in T (not sig) |

### Interpretation

The counter-narrative intervention produces a clear and somewhat paradoxical pattern:

1. **CNs increase exposure**: Replying to a harmful tweet with a CN generates activity that boosts the tweet's visibility in the platform's algorithm. Treatment tweets receive significantly more views — a consistent, robust finding across all analyses.

2. **CNs may reduce engagement propensity**: Despite increased visibility, Treatment tweets show a consistent (though not statistically significant) tendency to receive fewer Likes. The engagement rate (Likes/Views) is lower for Treatment, suggesting that while more people *see* the tweet, fewer choose to *endorse* it. This is consistent with the CN providing context that discourages positive engagement with the harmful narrative.

3. **Shares are unaffected**: The extreme zero-inflation (85% of tweets gain no Shares) leaves insufficient statistical power to detect any effect, regardless of whether one exists.

4. **The "visibility paradox"**: The CN simultaneously increases a tweet's reach while potentially reducing its persuasive impact per viewer. Whether this tradeoff is net-positive or net-negative depends on how one weighs exposure vs. endorsement — a question for the Discussion section.

# 4. Moderator & Heterogeneity Analysis

## Overview

Section 3 established the **average** treatment effect:
- **Views:** Significant anti-suppression (Treatment grew MORE)
- **Likes:** Directional suppression (not significant)
- **Shares:** No detectable effect

Section 4 asks: **under what conditions are CNs more or less effective?**

We examine five potential moderators:
1. **Tweet Type** (post vs comment) — both groups
2. **Narrative Cluster** (6 thematic groups) — both groups
3. **Detection Lag** (response speed) — both groups
4. **Num_of_CNs** (CN dosage) — Treatment only
5. **KPI** (CN optimization target) — Treatment only

Sections 4.2–4.6 provide **descriptive moderator snapshots** (stratified effect sizes + forest plots).  
Section 4.7 provides the **unified multivariate analysis** via XGBoost + SHAP.

**Important caveat:** The randomization supports the main C vs T comparison (Section 3). Moderator effects are observational — narratives, detection lag, and CN characteristics were not experimentally varied. These findings are hypothesis-generating.

## 4.1: Data Preparation & Factor Exploration

### Goal

Before running any moderator analysis, we need to:
1. Map narratives to the 6 pre-defined thematic clusters
2. Merge any missing columns from the original data (KPI, Num_of_CNs are already in complete_df)
3. Explore all factor distributions and sample sizes
4. Decide which analyses are adequately powered

### 4.1.1: Map Narratives to Clusters

Six thematic clusters were pre-defined based on the narrative content:

| Cluster | Label | Narratives |
|---------|-------|------------|
| 1 | NATO / Western Aggression & Broken Promises | NATO Expansion, Western Aggression, Ukraine Coup, Broken NATO Promises, Western Imperialism |
| 2 | Western Manipulation, Hegemony & Moral Decay | Color Revolutions, Anti-Russian Propaganda, Peaceful Resolution, US Sovereignty, Traditional Values |
| 3 | Humanitarian / 'Denazification' & Ukraine's Wrongdoing | Ukraine Nazis, Protecting Donbas, Persecutes Minorities, Weapons Recklessly, US Funds Neo-Nazis |
| 4 | Military Success & Liberation | Military Success, Liberating Territories |
| 5 | Territorial Legitimacy via Referendum | Crimea |
| 6 | Economic Warfare & War-Profit Claims | Prolonging Conflict for Profit, Sanctions Ineffective |

In [ ]:
# --------------------------------------------------
# 4.1.1: MAP NARRATIVES TO CLUSTERS
# --------------------------------------------------

# Define the mapping from individual narratives to thematic clusters
NARRATIVE_CLUSTER_MAP = {
    # Cluster 1: NATO / Western Aggression & Broken Promises
    'NATO Expansion Provoked Russian Invasion': 1,
    'Russia Defending Against Western Aggression': 1,
    'West Orchestrated Ukraine Coup in 2014': 1,
    'West Broke Promises About NATO': 1,
    'Russia Defending Against Western Imperialism': 1,

    # Cluster 2: Western Manipulation, Hegemony & Moral Decay
    'Color Revolutions are US-Backed Coups': 2,
    'West Spreads Anti-Russian Propaganda': 2,
    'Russia Seeks Peaceful Conflict Resolution': 2,
    'US Undermines Global Sovereignty': 2,
    'Russian World Defending Traditional Values': 2,

    # Cluster 3: Humanitarian / 'Denazification' & Ukraine's Wrongdoing
    'Ukraine Controlled by Nazis': 3,
    'Protecting Ethnic Russians in Donbas': 3,
    'Ukraine Persecutes Russian-Speaking Minorities': 3,
    'Ukraine Using Western Weapons Recklessly': 3,
    'US Funds Neo-Nazi Groups in Ukraine': 3,

    # Cluster 4: Military Success & Liberation
    'Russian Military Achieving Successful Operation': 4,
    'Russia Liberating Ukrainian Territories': 4,

    # Cluster 5: Territorial Legitimacy via Referendum
    'Crimea Legitimately Part of Russia': 5,

    # Cluster 6: Economic Warfare & War-Profit Claims
    'West Prolonging Conflict for Profit': 6,
    'Sanctions Ineffective Against Russia': 6,
}

CLUSTER_LABELS = {
    1: 'NATO/Western Aggression',
    2: 'Western Manipulation',
    3: 'Denazification/Ukraine',
    4: 'Military Success',
    5: 'Crimea Referendum',
    6: 'Economic Warfare',
}

# Apply mapping
complete_df['Narrative_Cluster'] = complete_df['Narrative'].map(NARRATIVE_CLUSTER_MAP)
complete_df['Cluster_Label'] = complete_df['Narrative_Cluster'].map(CLUSTER_LABELS)

# Check for unmapped narratives
unmapped = complete_df[complete_df['Narrative_Cluster'].isna()]['Narrative'].unique()
if len(unmapped) > 0:
    print(f"⚠️ UNMAPPED NARRATIVES: {unmapped}")
else:
    print("✅ All narratives mapped successfully")

# Show cluster distribution
print("\n=== NARRATIVE CLUSTER DISTRIBUTION ===\n")
cluster_dist = complete_df.groupby(['Narrative_Cluster', 'Cluster_Label'])['Group'].value_counts().unstack(fill_value=0)
cluster_dist['Total'] = cluster_dist.sum(axis=1)
cluster_dist['C%'] = (cluster_dist['Control'] / cluster_dist['Total'] * 100).round(1)
cluster_dist = cluster_dist.sort_values('Total', ascending=False)
print(cluster_dist.to_string())

# Show which narratives went into each cluster
print("\n\n=== NARRATIVES PER CLUSTER ===\n")
for cid in sorted(CLUSTER_LABELS.keys()):
    label = CLUSTER_LABELS[cid]
    narratives = [k for k, v in NARRATIVE_CLUSTER_MAP.items() if v == cid]
    subset = complete_df[complete_df['Narrative_Cluster'] == cid]
    print(f"Cluster {cid}: {label} (n={len(subset)})")
    for narr in narratives:
        n = len(subset[subset['Narrative'] == narr])
        if n > 0:
            print(f"    {narr}: {n}")
    print()

### 4.1.2: Explore Treatment-Only Factors

KPI and Num_of_CNs only exist for Treatment tweets. We need to understand their distributions before deciding how to analyze them.

In [ ]:
# --------------------------------------------------
# 4.1.2: EXPLORE TREATMENT-ONLY FACTORS
# --------------------------------------------------

trt = complete_df[complete_df['Group'] == 'Treatment']

# --- KPI ---
print("=== KPI DISTRIBUTION (Treatment only) ===\n")
kpi_dist = trt['KPI'].value_counts(dropna=False)
print(kpi_dist.to_string())
print(f"\nUnique KPI values: {trt['KPI'].nunique(dropna=False)}")
print(f"Missing/None: {trt['KPI'].isna().sum()}")

# --- Num_of_CNs ---
print("\n\n=== NUM_OF_CNs DISTRIBUTION (Treatment only) ===\n")
cn_dist = trt['Num_of_CNs'].value_counts().sort_index()
print(cn_dist.to_string())
print(f"\nRange: {trt['Num_of_CNs'].min()} to {trt['Num_of_CNs'].max()}")
print(f"Mean: {trt['Num_of_CNs'].mean():.2f}, Median: {trt['Num_of_CNs'].median():.0f}")

# --- Detection Lag ---
print("\n\n=== DETECTION LAG (Both groups) ===\n")
for grp_name, grp_df in complete_df.groupby('Group'):
    lag = grp_df['Detection_Lag_Minutes']
    print(f"{grp_name}: mean={lag.mean():.1f} min, median={lag.median():.1f} min, "
          f"min={lag.min():.1f}, max={lag.max():.1f}, std={lag.std():.1f}")

# Terciles for later use
lag_terciles = complete_df['Detection_Lag_Minutes'].quantile([0.333, 0.667])
print(f"\nTercile boundaries: Fast < {lag_terciles.iloc[0]:.1f} min, "
      f"Medium < {lag_terciles.iloc[1]:.1f} min, Slow ≥ {lag_terciles.iloc[1]:.1f} min")

complete_df['Lag_Tercile'] = pd.cut(
    complete_df['Detection_Lag_Minutes'],
    bins=[-np.inf, lag_terciles.iloc[0], lag_terciles.iloc[1], np.inf],
    labels=['Fast', 'Medium', 'Slow']
)
print("\nLag tercile distribution:")
print(complete_df.groupby(['Lag_Tercile', 'Group']).size().unstack(fill_value=0).to_string())

### 4.1.3: Sample Size & Power Assessment

For each factor × group combination, we need sufficient sample size for meaningful comparison. Rule of thumb: n ≥ 30 per cell for rank-based tests.

In [ ]:
# --------------------------------------------------
# 4.1.3: SAMPLE SIZE TABLE
# --------------------------------------------------

print("=== SAMPLE SIZE PER CELL: BOTH-GROUP FACTORS ===\n")
print("(Need n ≥ 30 per cell for reliable rank-based tests)\n")

# Tweet Type
print("--- Tweet Type × Group ---")
type_sizes = complete_df.groupby(['Type', 'Group']).size().unstack(fill_value=0)
type_sizes['Total'] = type_sizes.sum(axis=1)
type_sizes['Powered?'] = type_sizes[['Control', 'Treatment']].min(axis=1).apply(
    lambda x: '✅' if x >= 30 else '⚠️')
print(type_sizes.to_string())

# Narrative Cluster
print("\n--- Narrative Cluster × Group ---")
cluster_sizes = complete_df.groupby(['Cluster_Label', 'Group']).size().unstack(fill_value=0)
cluster_sizes['Total'] = cluster_sizes.sum(axis=1)
cluster_sizes['Powered?'] = cluster_sizes[['Control', 'Treatment']].min(axis=1).apply(
    lambda x: '✅' if x >= 30 else '⚠️')
cluster_sizes = cluster_sizes.sort_values('Total', ascending=False)
print(cluster_sizes.to_string())

# Detection Lag Tercile
print("\n--- Detection Lag Tercile × Group ---")
lag_sizes = complete_df.groupby(['Lag_Tercile', 'Group']).size().unstack(fill_value=0)
lag_sizes['Total'] = lag_sizes.sum(axis=1)
lag_sizes['Powered?'] = lag_sizes[['Control', 'Treatment']].min(axis=1).apply(
    lambda x: '✅' if x >= 30 else '⚠️')
print(lag_sizes.to_string())

print("\n\n=== SAMPLE SIZE: TREATMENT-ONLY FACTORS ===\n")
print("(Need n ≥ 30 per level for reliable comparison)\n")

trt = complete_df[complete_df['Group'] == 'Treatment']

# Num_of_CNs
print("--- Num_of_CNs ---")
cn_sizes = trt['Num_of_CNs'].value_counts().sort_index().reset_index()
cn_sizes.columns = ['Num_of_CNs', 'n']
cn_sizes['Powered?'] = cn_sizes['n'].apply(lambda x: '✅' if x >= 30 else '⚠️')
print(cn_sizes.to_string(index=False))

# KPI
print("\n--- KPI ---")
kpi_sizes = trt['KPI'].value_counts(dropna=False).reset_index()
kpi_sizes.columns = ['KPI', 'n']
kpi_sizes['Powered?'] = kpi_sizes['n'].apply(lambda x: '✅' if x >= 30 else '⚠️')
print(kpi_sizes.to_string(index=False))

### 4.1.4: Analysis Decision Matrix

Based on the sample sizes above, here is the decision for each planned analysis. This cell will be updated after running 4.1.2 and 4.1.3 to reflect actual data.

In [ ]:
# --------------------------------------------------
# 4.1.4: DECISION MATRIX
# --------------------------------------------------
# This prints a summary of which analyses are feasible based on the
# sample sizes computed above. Review and adjust the plan accordingly.

print("=== ANALYSIS DECISION MATRIX ===\n")
print(f"{'Section':<12} {'Factor':<28} {'Design':<18} {'Status'}")
print("-" * 75)

# Tweet Type — always powered (comment > 400 per group, post > 100 per group)
print(f"{'4.2':<12} {'Tweet Type':<28} {'Both-group':<18} ✅ Proceed")

# Narrative Cluster — check min cell
min_cluster = cluster_sizes[['Control', 'Treatment']].min().min()
if min_cluster >= 30:
    print(f"{'4.3':<12} {'Narrative Cluster':<28} {'Both-group':<18} ✅ Proceed (all clusters powered)")
else:
    underpowered = cluster_sizes[cluster_sizes[['Control', 'Treatment']].min(axis=1) < 30].index.tolist()
    print(f"{'4.3':<12} {'Narrative Cluster':<28} {'Both-group':<18} ⚠️ Proceed with caveat")
    print(f"{'':12} Underpowered clusters: {underpowered}")

# Detection Lag — always powered (terciles by construction ~200 per cell)
print(f"{'4.4':<12} {'Detection Lag':<28} {'Both-group':<18} ✅ Proceed")

# Num_of_CNs — depends on distribution
cn_levels_powered = cn_sizes[cn_sizes['n'] >= 30]['Num_of_CNs'].tolist()
cn_levels_all = cn_sizes['Num_of_CNs'].tolist()
if len(cn_levels_powered) >= 2:
    print(f"{'4.5':<12} {'Num_of_CNs':<28} {'Treatment-only':<18} ✅ Proceed (levels with n≥30: {cn_levels_powered})")
else:
    print(f"{'4.5':<12} {'Num_of_CNs':<28} {'Treatment-only':<18} ⚠️ Limited — may need to bin")

# KPI — depends on distribution
kpi_powered = kpi_sizes[kpi_sizes['n'] >= 30]['KPI'].tolist()
if len(kpi_powered) >= 2:
    print(f"{'4.6':<12} {'KPI':<28} {'Treatment-only':<18} ✅ Proceed (categories with n≥30: {kpi_powered})")
elif len(kpi_powered) == 1:
    print(f"{'4.6':<12} {'KPI':<28} {'Treatment-only':<18} ⚠️ Only 1 powered category — descriptive only")
else:
    print(f"{'4.6':<12} {'KPI':<28} {'Treatment-only':<18} ❌ Skip — insufficient data")

print(f"{'4.7':<12} {'XGBoost + SHAP':<28} {'Unified':<18} ✅ Proceed (uses all features)")

# Update ctrl and trt for later sections
ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

print(f"\n\n=== DATA READY FOR SECTION 4 ===")
print(f"Total: {len(complete_df)} tweets (Control: {len(ctrl)}, Treatment: {len(trt)})")
print(f"New columns added: Narrative_Cluster, Cluster_Label, Lag_Tercile")

## 4.2: Tweet Type — Post vs Comment

### Motivation

The dataset contains two tweet types: **posts** (standalone tweets, n=228) and **comments** (replies to other tweets, n=982). These have fundamentally different visibility dynamics on X:

- **Posts** appear directly in followers' timelines and are the root of threads. A CN reply to a post is the first (or among the first) replies — highly visible.
- **Comments** are nested in existing threads. A CN reply to a comment may be buried several levels deep — much less visible.

This structural difference could moderate the CN effect. If CN visibility drives the effect, we'd expect stronger effects for posts than comments.

### Approach

For each tweet type, we compute the same core battery from Section 3.2 (rank-biserial r, MWU, medians/trimmed means), then visualize all effect sizes in a single forest plot.

In [ ]:
# --------------------------------------------------
# 4.2: TWEET TYPE — POST vs COMMENT
# --------------------------------------------------

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

METRICS = ['Views', 'Likes', 'Shares']
TYPES = ['post', 'comment']

ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

# Store results for forest plot
forest_results = []

print("=" * 90)
print("4.2: STRATIFIED ANALYSIS BY TWEET TYPE")
print("=" * 90)

for tweet_type in TYPES:
    c = ctrl[ctrl['Type'] == tweet_type]
    t = trt[trt['Type'] == tweet_type]

    print(f"\n{'─' * 90}")
    print(f"  TWEET TYPE: {tweet_type.upper()} (Control n={len(c)}, Treatment n={len(t)})")
    print(f"{'─' * 90}")
    print(f"  {'Metric':<8} {'C Med':>8} {'T Med':>8} {'C TrMean':>10} {'T TrMean':>10} "
          f"{'r':>7} {'p':>8} {'Sig':>4}")
    print(f"  {'-'*72}")

    for metric in METRICS:
        col = f'{metric}_Growth_13d'
        cv = c[col].dropna()
        tv = t[col].dropna()

        # Rank-biserial r
        mwu_stat, mwu_p = stats.mannwhitneyu(cv, tv)
        n1, n2 = len(cv), len(tv)
        r = (2 * mwu_stat) / (n1 * n2) - 1

        # Medians and trimmed means
        c_med = cv.median()
        t_med = tv.median()
        c_tm = stats.trim_mean(cv.values, 0.05)
        t_tm = stats.trim_mean(tv.values, 0.05)

        sig = '*' if mwu_p < 0.05 else ''
        print(f"  {metric:<8} {c_med:>7.1f}% {t_med:>7.1f}% {c_tm:>9.1f}% {t_tm:>9.1f}% "
              f"{r:>+7.4f} {mwu_p:>8.4f} {sig:>4}")

        # Bootstrap CI for r
        np.random.seed(42)
        r_boots = []
        combined = np.concatenate([cv.values, tv.values])
        for _ in range(5000):
            idx_c = np.random.choice(n1, n1, replace=True)
            idx_t = np.random.choice(n2, n2, replace=True)
            boot_c = cv.values[idx_c]
            boot_t = tv.values[idx_t]
            boot_stat, _ = stats.mannwhitneyu(boot_c, boot_t)
            r_boots.append((2 * boot_stat) / (n1 * n2) - 1)

        r_ci_lo = np.percentile(r_boots, 2.5)
        r_ci_hi = np.percentile(r_boots, 97.5)

        forest_results.append({
            'metric': metric,
            'subgroup': tweet_type,
            'r': r,
            'ci_lo': r_ci_lo,
            'ci_hi': r_ci_hi,
            'p': mwu_p,
            'n_ctrl': n1,
            'n_trt': n2,
        })

# Also compute overall effect for reference line
print(f"\n{'─' * 90}")
print(f"  OVERALL (reference from Section 3)")
print(f"{'─' * 90}")
print(f"  {'Metric':<8} {'r':>7} {'p':>8}")
print(f"  {'-'*20}")
for metric in METRICS:
    col = f'{metric}_Growth_13d'
    cv = ctrl[col].dropna()
    tv = trt[col].dropna()
    mwu_stat, mwu_p = stats.mannwhitneyu(cv, tv)
    r = (2 * mwu_stat) / (len(cv) * len(tv)) - 1
    print(f"  {metric:<8} {r:>+7.4f} {mwu_p:>8.4f}")

### 4.2.1: Forest Plot — Effect Size by Tweet Type

Each point shows the rank-biserial r (with 95% bootstrap CI) for the C vs T comparison within that tweet type. The dashed vertical line marks r = 0 (no effect). Positive r = suppression direction (Control grew more); negative r = anti-suppression (Treatment grew more). The diamond markers show the overall effect from Section 3 for reference.


In [ ]:
# --------------------------------------------------
# 4.2.1: FOREST PLOT — TWEET TYPE
# --------------------------------------------------

import pandas as pd

res_df = pd.DataFrame(forest_results)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

colors = {'post': '#e74c3c', 'comment': '#3498db'}

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    sub = res_df[res_df['metric'] == metric].reset_index(drop=True)

    y_positions = list(range(len(sub)))

    for i, row in sub.iterrows():
        color = colors[row['subgroup']]
        marker = 'o' if row['p'] < 0.05 else 'o'
        facecolor = color if row['p'] < 0.05 else 'white'

        ax.errorbar(row['r'], i, xerr=[[row['r'] - row['ci_lo']], [row['ci_hi'] - row['r']]],
                    fmt='o', color=color, markerfacecolor=facecolor, markeredgecolor=color,
                    markersize=10, capsize=5, linewidth=2, capthick=1.5,
                    label=f"{row['subgroup']} (n={row['n_ctrl']}+{row['n_trt']})")

    # Overall effect as diamond
    col = f'{metric}_Growth_13d'
    cv_all = ctrl[col].dropna()
    tv_all = trt[col].dropna()
    mwu_all, _ = stats.mannwhitneyu(cv_all, tv_all)
    r_all = (2 * mwu_all) / (len(cv_all) * len(tv_all)) - 1
    ax.plot(r_all, -0.5, 'D', color='black', markersize=10, label='Overall', zorder=5)

    ax.axvline(0, color='grey', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_yticks(y_positions)
    ax.set_yticklabels([row['subgroup'].title() for _, row in sub.iterrows()])
    ax.set_xlabel('Rank-biserial r')
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.legend(fontsize=8, loc='best')

    # Shade suppression vs anti-suppression regions
    xlim = ax.get_xlim()
    ax.axvspan(0, max(xlim[1], 0.3), alpha=0.04, color='red')    # suppression
    ax.axvspan(min(xlim[0], -0.3), 0, alpha=0.04, color='green')  # anti-suppression
    ax.text(max(xlim[1], 0.2) * 0.7, len(sub) - 0.3, 'Suppression →', fontsize=7,
            color='red', alpha=0.5, ha='center')
    ax.text(min(xlim[0], -0.2) * 0.7, len(sub) - 0.3, '← Anti-supp.', fontsize=7,
            color='green', alpha=0.5, ha='center')

fig.suptitle('4.2: Treatment Effect by Tweet Type (13-day growth)', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_2_forest_type.png', dpi=150, bbox_inches='tight')
plt.show()
# --------------------------------------------------
# 4.2 AUTOMATED TAKEAWAYS
# --------------------------------------------------

print("=" * 70)
print("4.2 TAKEAWAYS: Tweet Type as Moderator")
print("=" * 70)

res_df = pd.DataFrame(forest_results)

for metric in METRICS:
    sub = res_df[res_df['metric'] == metric]
    post_r = sub[sub['subgroup'] == 'post'].iloc[0]
    comment_r = sub[sub['subgroup'] == 'comment'].iloc[0]

    diff = abs(post_r['r'] - comment_r['r'])
    stronger = 'post' if abs(post_r['r']) > abs(comment_r['r']) else 'comment'

    print(f"\n{metric}:")
    print(f"  Post:    r = {post_r['r']:+.4f}  [{post_r['ci_lo']:+.4f}, {post_r['ci_hi']:+.4f}]  "
          f"p = {post_r['p']:.4f} {'*' if post_r['p'] < 0.05 else ''}")
    print(f"  Comment: r = {comment_r['r']:+.4f}  [{comment_r['ci_lo']:+.4f}, {comment_r['ci_hi']:+.4f}]  "
          f"p = {comment_r['p']:.4f} {'*' if comment_r['p'] < 0.05 else ''}")

    # Check if CIs overlap (crude interaction test)
    overlap = not (post_r['ci_hi'] < comment_r['ci_lo'] or comment_r['ci_hi'] < post_r['ci_lo'])

    if overlap:
        print(f"  → CIs overlap — no significant difference between post and comment effects")
    else:
        print(f"  → CIs do NOT overlap — tweet type significantly moderates the effect")
        print(f"    Effect is stronger for {stronger} (|Δr| = {diff:.4f})")

print("\n" + "=" * 70)
print("Note: Filled markers in the forest plot = p < 0.05; hollow = not significant.")
print("CI overlap is a conservative test for interaction; formal test in 4.7 (XGBoost).")
print("=" * 70)

## 4.3: Narrative Cluster

### Motivation

The 20 disinformation narratives in this experiment were grouped into 6 thematic clusters. Different narratives may be more or less susceptible to counter-narrative intervention — a "NATO expansion" claim may respond differently to factual rebuttal than a "Ukraine has Nazis" emotional claim.

This is among the most **policy-relevant** moderator questions: if certain narrative families resist CN intervention, resources should be redirected toward narratives where CNs are effective.

### Sample sizes

| Cluster | n (C + T) | Powered? |
|---------|-----------|----------|
| 1. NATO/Western Aggression | 638 (325 + 313) | ✅ |
| 2. Western Manipulation | 133 (67 + 66) | ✅ |
| 3. Denazification/Ukraine | 212 (110 + 102) | ✅ |
| 4. Military Success | 119 (61 + 58) | ✅ |
| 5. Crimea Referendum | 41 (22 + 19) | ⚠️ |
| 6. Economic Warfare | 67 (33 + 34) | ✅ |

Cluster 5 (Crimea) has only 41 tweets. We include it for completeness but flag wide CIs.

In [ ]:
# --------------------------------------------------
# 4.3: NARRATIVE CLUSTER — STRATIFIED ANALYSIS
# --------------------------------------------------

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

METRICS = ['Views', 'Likes', 'Shares']

ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

# Ordered by sample size for consistent display
CLUSTER_ORDER = [1, 3, 2, 4, 6, 5]

forest_results = []

print("=" * 95)
print("4.3: STRATIFIED ANALYSIS BY NARRATIVE CLUSTER")
print("=" * 95)

for cid in CLUSTER_ORDER:
    label = CLUSTER_LABELS[cid]
    c = ctrl[ctrl['Narrative_Cluster'] == cid]
    t = trt[trt['Narrative_Cluster'] == cid]

    flag = ' ⚠️ UNDERPOWERED' if min(len(c), len(t)) < 30 else ''

    print(f"\n{'─' * 95}")
    print(f"  CLUSTER {cid}: {label} (Control n={len(c)}, Treatment n={len(t)}){flag}")
    print(f"{'─' * 95}")
    print(f"  {'Metric':<8} {'C Med':>8} {'T Med':>8} {'C TrMean':>10} {'T TrMean':>10} "
          f"{'r':>7} {'p':>8} {'Sig':>4}")
    print(f"  {'-'*72}")

    for metric in METRICS:
        col = f'{metric}_Growth_13d'
        cv = c[col].dropna()
        tv = t[col].dropna()

        if len(cv) < 5 or len(tv) < 5:
            print(f"  {metric:<8} {'— insufficient data —'}")
            continue

        # Rank-biserial r
        mwu_stat, mwu_p = stats.mannwhitneyu(cv, tv)
        n1, n2 = len(cv), len(tv)
        r = (2 * mwu_stat) / (n1 * n2) - 1

        # Medians and trimmed means
        c_med = cv.median()
        t_med = tv.median()
        c_tm = stats.trim_mean(cv.values, 0.05)
        t_tm = stats.trim_mean(tv.values, 0.05)

        sig = '*' if mwu_p < 0.05 else ''
        print(f"  {metric:<8} {c_med:>7.1f}% {t_med:>7.1f}% {c_tm:>9.1f}% {t_tm:>9.1f}% "
              f"{r:>+7.4f} {mwu_p:>8.4f} {sig:>4}")

        # Bootstrap CI for r
        np.random.seed(42 + cid * 10 + METRICS.index(metric))
        r_boots = []
        for _ in range(5000):
            idx_c = np.random.choice(n1, n1, replace=True)
            idx_t = np.random.choice(n2, n2, replace=True)
            boot_c = cv.values[idx_c]
            boot_t = tv.values[idx_t]
            boot_stat, _ = stats.mannwhitneyu(boot_c, boot_t)
            r_boots.append((2 * boot_stat) / (n1 * n2) - 1)

        r_ci_lo = np.percentile(r_boots, 2.5)
        r_ci_hi = np.percentile(r_boots, 97.5)

        forest_results.append({
            'metric': metric,
            'cluster_id': cid,
            'subgroup': label,
            'r': r,
            'ci_lo': r_ci_lo,
            'ci_hi': r_ci_hi,
            'p': mwu_p,
            'n_ctrl': n1,
            'n_trt': n2,
            'underpowered': min(n1, n2) < 30,
        })

### 4.3.1: Forest Plot — Effect Size by Narrative Cluster

Each row is one narrative cluster. Filled markers = p < 0.05; hollow = not significant. The ⚠️ marker indicates an underpowered cluster (Crimea, n=41). Black diamond = overall effect from Section 3.


In [ ]:
# --------------------------------------------------
# 4.3.1: FOREST PLOT — NARRATIVE CLUSTER
# --------------------------------------------------

res_df = pd.DataFrame(forest_results)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

# Color palette for clusters
cluster_colors = {
    1: '#2c3e50', 2: '#8e44ad', 3: '#c0392b',
    4: '#27ae60', 5: '#f39c12', 6: '#2980b9'
}

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    sub = res_df[res_df['metric'] == metric].reset_index(drop=True)

    # Sort by cluster order (largest first, Crimea last)
    sub = sub.set_index('cluster_id').loc[CLUSTER_ORDER].reset_index()

    y_positions = list(range(len(sub)))

    for i, row in sub.iterrows():
        color = cluster_colors[row['cluster_id']]
        facecolor = color if row['p'] < 0.05 else 'white'
        marker = 'o'
        ms = 10

        # Special marker for underpowered
        if row['underpowered']:
            marker = 's'  # square for underpowered
            ms = 9

        ax.errorbar(row['r'], i,
                    xerr=[[row['r'] - row['ci_lo']], [row['ci_hi'] - row['r']]],
                    fmt=marker, color=color, markerfacecolor=facecolor, markeredgecolor=color,
                    markersize=ms, capsize=4, linewidth=1.8, capthick=1.3)

    # Overall effect as diamond
    col = f'{metric}_Growth_13d'
    cv_all = ctrl[col].dropna()
    tv_all = trt[col].dropna()
    mwu_all, _ = stats.mannwhitneyu(cv_all, tv_all)
    r_all = (2 * mwu_all) / (len(cv_all) * len(tv_all)) - 1
    ax.plot(r_all, -0.7, 'D', color='black', markersize=10, zorder=5)

    ax.axvline(0, color='grey', linestyle='--', linewidth=1, alpha=0.7)

    # Y-axis labels with sample sizes
    ylabels = []
    for _, row in sub.iterrows():
        flag = ' ⚠️' if row['underpowered'] else ''
        ylabels.append(f"{row['subgroup']}{flag}\n(n={row['n_ctrl']}+{row['n_trt']})")

    ax.set_yticks(y_positions)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xlabel('Rank-biserial r', fontsize=11)
    ax.set_title(metric, fontweight='bold', fontsize=13)

    # Shade regions
    xlim = ax.get_xlim()
    ax.axvspan(0, max(xlim[1], 0.3), alpha=0.04, color='red')
    ax.axvspan(min(xlim[0], -0.3), 0, alpha=0.04, color='green')

    ax.invert_yaxis()

# Add legend manually
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='grey', markerfacecolor='grey', markersize=8, linestyle='None', label='p < 0.05'),
    Line2D([0], [0], marker='o', color='grey', markerfacecolor='white', markeredgecolor='grey', markersize=8, linestyle='None', label='p ≥ 0.05'),
    Line2D([0], [0], marker='s', color='grey', markerfacecolor='white', markeredgecolor='grey', markersize=8, linestyle='None', label='Underpowered (n<30/cell)'),
    Line2D([0], [0], marker='D', color='black', markersize=8, linestyle='None', label='Overall effect'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('4.3: Treatment Effect by Narrative Cluster (13-day growth)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_3_forest_narrative.png', dpi=150, bbox_inches='tight')
plt.show()
# --------------------------------------------------
# 4.3 AUTOMATED TAKEAWAYS
# --------------------------------------------------

res_df = pd.DataFrame(forest_results)

print("=" * 75)
print("4.3 TAKEAWAYS: Narrative Cluster as Moderator")
print("=" * 75)

for metric in METRICS:
    sub = res_df[res_df['metric'] == metric].sort_values('r')

    print(f"\n{'─' * 75}")
    print(f"  {metric}")
    print(f"{'─' * 75}")

    sig_clusters = sub[sub['p'] < 0.05]

    for _, row in sub.iterrows():
        flag = ' ⚠️' if row['underpowered'] else ''
        sig = ' *' if row['p'] < 0.05 else ''
        print(f"  {row['subgroup']:<28}{flag}  r = {row['r']:+.4f}  "
              f"[{row['ci_lo']:+.4f}, {row['ci_hi']:+.4f}]  p = {row['p']:.4f}{sig}")

    if len(sig_clusters) > 0:
        print(f"\n  Significant clusters: {', '.join(sig_clusters['subgroup'].tolist())}")
    else:
        print(f"\n  No individual cluster reaches significance for {metric}")

    # Range of effect sizes
    r_range = sub['r'].max() - sub['r'].min()
    most_supp = sub.iloc[-1]   # highest r = most suppression
    most_anti = sub.iloc[0]    # lowest r = most anti-suppression
    print(f"  Effect size range: {r_range:.4f} (most anti-supp: {most_anti['subgroup']}, "
          f"most supp: {most_supp['subgroup']})")

# Cross-cluster consistency check
print(f"\n{'=' * 75}")
print("CROSS-METRIC PATTERNS")
print(f"{'=' * 75}")
print("\nDo any clusters show QUALITATIVELY different patterns from the overall?")
print("(i.e., suppression for Views where overall shows anti-suppression, or vice versa)\n")

for cid in CLUSTER_ORDER:
    label = CLUSTER_LABELS[cid]
    sub = res_df[res_df['cluster_id'] == cid]
    views_r = sub[sub['metric'] == 'Views']['r'].values[0]
    likes_r = sub[sub['metric'] == 'Likes']['r'].values[0]

    pattern = f"Views r={views_r:+.3f}, Likes r={likes_r:+.3f}"

    # Flag if Views is positive (suppression) — opposite to overall
    anomaly = ""
    if views_r > 0.05:
        anomaly = " ← ANOMALY: Views suppressed (opposite to overall)"
    if likes_r < -0.05:
        anomaly = " ← ANOMALY: Likes anti-suppressed (opposite to overall)"

    print(f"  {label:<28} {pattern}{anomaly}")

## 4.4: Detection Lag (Response Speed)

### Motivation

Detection Lag measures the time (in minutes) between when a harmful tweet was created and when our system detected it. Since CN deployment follows detection, this serves as a proxy for **CN response speed** — faster detection ≈ faster CN.

The hypothesis is straightforward: **CNs deployed earlier should be more effective**, because they arrive before the tweet gains traction and shapes audience perception. If a tweet has already gone viral by the time the CN arrives, the CN may be too late.

This is directly actionable — if speed matters, it justifies investment in faster detection systems.

### Operationalization

Detection_Lag_Minutes was split into terciles:
- **Fast:** < ~42 minutes
- **Medium:** ~42–109 minutes  
- **Slow:** ≥ ~109 minutes

Each tercile has ~200 tweets per group (well-powered).

In [ ]:
# --------------------------------------------------
# 4.4: DETECTION LAG — STRATIFIED ANALYSIS
# --------------------------------------------------

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

METRICS = ['Views', 'Likes', 'Shares']
LAG_ORDER = ['Fast', 'Medium', 'Slow']

ctrl = complete_df[complete_df['Group'] == 'Control']
trt = complete_df[complete_df['Group'] == 'Treatment']

forest_results = []

print("=" * 95)
print("4.4: STRATIFIED ANALYSIS BY DETECTION LAG (RESPONSE SPEED)")
print("=" * 95)

# Show tercile boundaries
lag_terciles = complete_df['Detection_Lag_Minutes'].quantile([0.333, 0.667])
print(f"\nTercile boundaries: Fast < {lag_terciles.iloc[0]:.1f} min | "
      f"Medium: {lag_terciles.iloc[0]:.1f}–{lag_terciles.iloc[1]:.1f} min | "
      f"Slow ≥ {lag_terciles.iloc[1]:.1f} min")

for lag_bin in LAG_ORDER:
    c = ctrl[ctrl['Lag_Tercile'] == lag_bin]
    t = trt[trt['Lag_Tercile'] == lag_bin]

    # Show lag range for this bin
    bin_data = complete_df[complete_df['Lag_Tercile'] == lag_bin]['Detection_Lag_Minutes']

    print(f"\n{'─' * 95}")
    print(f"  {lag_bin.upper()} RESPONSE (Control n={len(c)}, Treatment n={len(t)}) "
          f"— Lag: {bin_data.min():.0f}–{bin_data.max():.0f} min, median={bin_data.median():.0f} min")
    print(f"{'─' * 95}")
    print(f"  {'Metric':<8} {'C Med':>8} {'T Med':>8} {'C TrMean':>10} {'T TrMean':>10} "
          f"{'r':>7} {'p':>8} {'Sig':>4}")
    print(f"  {'-'*72}")

    for metric in METRICS:
        col = f'{metric}_Growth_13d'
        cv = c[col].dropna()
        tv = t[col].dropna()

        # Rank-biserial r
        mwu_stat, mwu_p = stats.mannwhitneyu(cv, tv)
        n1, n2 = len(cv), len(tv)
        r = (2 * mwu_stat) / (n1 * n2) - 1

        c_med = cv.median()
        t_med = tv.median()
        c_tm = stats.trim_mean(cv.values, 0.05)
        t_tm = stats.trim_mean(tv.values, 0.05)

        sig = '*' if mwu_p < 0.05 else ''
        print(f"  {metric:<8} {c_med:>7.1f}% {t_med:>7.1f}% {c_tm:>9.1f}% {t_tm:>9.1f}% "
              f"{r:>+7.4f} {mwu_p:>8.4f} {sig:>4}")

        # Bootstrap CI
        np.random.seed(42 + LAG_ORDER.index(lag_bin) * 100 + METRICS.index(metric))
        r_boots = []
        for _ in range(5000):
            idx_c = np.random.choice(n1, n1, replace=True)
            idx_t = np.random.choice(n2, n2, replace=True)
            boot_stat, _ = stats.mannwhitneyu(cv.values[idx_c], tv.values[idx_t])
            r_boots.append((2 * boot_stat) / (n1 * n2) - 1)

        forest_results.append({
            'metric': metric,
            'subgroup': lag_bin,
            'r': r,
            'ci_lo': np.percentile(r_boots, 2.5),
            'ci_hi': np.percentile(r_boots, 97.5),
            'p': mwu_p,
            'n_ctrl': n1,
            'n_trt': n2,
        })

### 4.4.1: Forest Plot — Effect Size by Response Speed

In [ ]:
# --------------------------------------------------
# 4.4.1: FOREST PLOT — DETECTION LAG
# --------------------------------------------------

res_df = pd.DataFrame(forest_results)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

lag_colors = {'Fast': '#27ae60', 'Medium': '#f39c12', 'Slow': '#e74c3c'}

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    sub = res_df[res_df['metric'] == metric]
    # Ensure order: Fast, Medium, Slow
    sub = sub.set_index('subgroup').loc[LAG_ORDER].reset_index()

    for i, row in sub.iterrows():
        color = lag_colors[row['subgroup']]
        facecolor = color if row['p'] < 0.05 else 'white'

        ax.errorbar(row['r'], i,
                    xerr=[[row['r'] - row['ci_lo']], [row['ci_hi'] - row['r']]],
                    fmt='o', color=color, markerfacecolor=facecolor, markeredgecolor=color,
                    markersize=10, capsize=5, linewidth=2, capthick=1.5,
                    label=f"{row['subgroup']} (n={row['n_ctrl']}+{row['n_trt']})")

    # Overall effect
    col = f'{metric}_Growth_13d'
    cv_all = ctrl[col].dropna()
    tv_all = trt[col].dropna()
    mwu_all, _ = stats.mannwhitneyu(cv_all, tv_all)
    r_all = (2 * mwu_all) / (len(cv_all) * len(tv_all)) - 1
    ax.plot(r_all, -0.5, 'D', color='black', markersize=10, zorder=5)

    ax.axvline(0, color='grey', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_yticks(range(len(LAG_ORDER)))
    ax.set_yticklabels(LAG_ORDER)
    ax.set_xlabel('Rank-biserial r')
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.legend(fontsize=8, loc='best')

    xlim = ax.get_xlim()
    ax.axvspan(0, max(xlim[1], 0.3), alpha=0.04, color='red')
    ax.axvspan(min(xlim[0], -0.3), 0, alpha=0.04, color='green')

fig.suptitle('4.4: Treatment Effect by Response Speed (13-day growth)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_4_forest_lag.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4.2: Scatter Plot — Detection Lag vs Growth

A continuous view: does growth correlate with detection lag differently for Control vs Treatment? LOWESS smoothers show the trend for each group.

In [ ]:
# --------------------------------------------------
# 4.4.2: SCATTER + LOWESS — DETECTION LAG vs GROWTH
# --------------------------------------------------

from statsmodels.nonparametric.smoothers_lowess import lowess

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    col = f'{metric}_Growth_13d'

    # Cap growth for visualization (keep actual data for LOWESS)
    cap = complete_df[col].quantile(0.95)

    for grp, color, label in [('Control', '#3498db', 'Control'), ('Treatment', '#e74c3c', 'Treatment')]:
        grp_data = complete_df[complete_df['Group'] == grp]
        x = grp_data['Detection_Lag_Minutes'].values
        y = grp_data[col].values

        # Plot points (capped for visibility)
        y_vis = np.clip(y, -100, cap)
        ax.scatter(x, y_vis, alpha=0.08, s=15, color=color, rasterized=True)

        # LOWESS smoother on uncapped data
        valid = ~np.isnan(x) & ~np.isnan(y)
        if valid.sum() > 50:
            smoothed = lowess(y[valid], x[valid], frac=0.4, return_sorted=True)
            # Clip smoother output for visualization
            smoothed_y = np.clip(smoothed[:, 1], -100, cap)
            ax.plot(smoothed[:, 0], smoothed_y, color=color, linewidth=3, label=label)

    ax.set_xlabel('Detection Lag (minutes)', fontsize=11)
    ax.set_ylabel(f'{metric} Growth % (13d)', fontsize=11)
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

    # Add tercile boundaries
    for q in lag_terciles.values:
        ax.axvline(q, color='grey', linestyle=':', alpha=0.4)

fig.suptitle('4.4: Detection Lag vs 13-day Growth (LOWESS smoothers)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_4_scatter_lag.png', dpi=150, bbox_inches='tight')
plt.show()
# --------------------------------------------------
# 4.4 AUTOMATED TAKEAWAYS
# --------------------------------------------------

res_df = pd.DataFrame(forest_results)

print("=" * 75)
print("4.4 TAKEAWAYS: Detection Lag (Response Speed) as Moderator")
print("=" * 75)

for metric in METRICS:
    sub = res_df[res_df['metric'] == metric].set_index('subgroup').loc[LAG_ORDER]

    print(f"\n  {metric}:")
    for lag_bin in LAG_ORDER:
        row = sub.loc[lag_bin]
        sig = ' *' if row['p'] < 0.05 else ''
        print(f"    {lag_bin:<8} r = {row['r']:+.4f}  [{row['ci_lo']:+.4f}, {row['ci_hi']:+.4f}]  "
              f"p = {row['p']:.4f}{sig}")

    # Check for monotonic trend: does r increase (more suppression) with slower response?
    r_vals = [sub.loc[b, 'r'] for b in LAG_ORDER]

    # Is there a clear gradient?
    if r_vals[0] < r_vals[1] < r_vals[2]:
        print(f"    → MONOTONIC TREND: Effect shifts toward suppression with slower response")
        print(f"      (Fast r={r_vals[0]:+.3f} → Medium r={r_vals[1]:+.3f} → Slow r={r_vals[2]:+.3f})")
    elif r_vals[0] > r_vals[1] > r_vals[2]:
        print(f"    → MONOTONIC TREND: Effect shifts toward anti-suppression with slower response")
        print(f"      (Fast r={r_vals[0]:+.3f} → Medium r={r_vals[1]:+.3f} → Slow r={r_vals[2]:+.3f})")
    else:
        print(f"    → No monotonic trend (Fast={r_vals[0]:+.3f}, Med={r_vals[1]:+.3f}, Slow={r_vals[2]:+.3f})")

# Overall conclusion
print(f"\n{'─' * 75}")
print("  SUMMARY: Does response speed matter?")
print(f"{'─' * 75}")
fast_sig = sum(1 for m in METRICS if res_df[(res_df['metric']==m) & (res_df['subgroup']=='Fast')].iloc[0]['p'] < 0.05)
slow_sig = sum(1 for m in METRICS if res_df[(res_df['metric']==m) & (res_df['subgroup']=='Slow')].iloc[0]['p'] < 0.05)
print(f"  Fast response: {fast_sig}/3 metrics significant")
print(f"  Slow response: {slow_sig}/3 metrics significant")
print(f"  If fast has more significant results → speed matters")
print(f"  If similar → speed does not moderate the treatment effect")

## 4.5: CN Dosage — Number of Counter-Narratives

### Motivation

Some Treatment tweets received multiple CN replies (1 to 5). If CNs work by providing visible counter-arguments, **more CNs might amplify the effect** (dose-response). Alternatively, there may be diminishing returns — one CN is enough, and additional ones add noise without benefit.

### Design

This is a **Treatment-only** analysis — we compare growth *within* the Treatment group across dosage levels. There is no Control comparison (Control tweets received 0 CNs by definition).

### Sample sizes

| Num_of_CNs | n | Powered? |
|------------|---|----------|
| 1 | 528 | ✅ |
| 2 | 18 | ⚠️ |
| 3 | 21 | ⚠️ |
| 4 | 11 | ⚠️ |
| 5 | 14 | ⚠️ |

89% of Treatment tweets received exactly 1 CN. We analyze both the full distribution (1–5) and a binned version (1 vs 2+).

In [ ]:
# --------------------------------------------------
# 4.5: CN DOSAGE — NUMBER OF COUNTER-NARRATIVES
# --------------------------------------------------

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

METRICS = ['Views', 'Likes', 'Shares']
trt = complete_df[complete_df['Group'] == 'Treatment'].copy()

# Create binned version
trt['CN_Bin'] = trt['Num_of_CNs'].apply(lambda x: '1 CN' if x == 1 else '2+ CNs')

print("=" * 90)
print("4.5: CN DOSAGE — TREATMENT-ONLY ANALYSIS")
print("=" * 90)

# --- Part A: Full distribution (1-5) ---
print("\n--- PART A: Full Distribution (Kruskal-Wallis across 1-5 CNs) ---\n")
print(f"  {'Metric':<8} {'KW stat':>8} {'p':>8} {'Sig':>4}   Medians by Num_of_CNs")
print(f"  {'-'*75}")

for metric in METRICS:
    col = f'{metric}_Growth_13d'
    groups = [trt[trt['Num_of_CNs'] == n][col].dropna().values for n in range(1, 6)]
    groups = [g for g in groups if len(g) >= 3]  # need at least 3 per group

    if len(groups) >= 2:
        kw_stat, kw_p = stats.kruskal(*groups)

        medians = []
        for n in range(1, 6):
            vals = trt[trt['Num_of_CNs'] == n][col]
            if len(vals) >= 3:
                medians.append(f"{n}CN:{vals.median():.0f}%")

        sig = '*' if kw_p < 0.05 else ''
        print(f"  {metric:<8} {kw_stat:>8.2f} {kw_p:>8.4f} {sig:>4}   {', '.join(medians)}")

# --- Part B: Binned (1 vs 2+) ---
print("\n\n--- PART B: Binned Comparison (1 CN vs 2+ CNs) ---\n")

cn1 = trt[trt['CN_Bin'] == '1 CN']
cn2p = trt[trt['CN_Bin'] == '2+ CNs']
print(f"  Sample sizes: 1 CN n={len(cn1)}, 2+ CNs n={len(cn2p)}")
print()

print(f"  {'Metric':<8} {'1CN Med':>9} {'2+ Med':>9} {'1CN TrMean':>12} {'2+ TrMean':>12} "
      f"{'r':>7} {'MWU p':>8} {'Sig':>4}")
print(f"  {'-'*78}")

bin_results = []
for metric in METRICS:
    col = f'{metric}_Growth_13d'
    v1 = cn1[col].dropna()
    v2 = cn2p[col].dropna()

    mwu_stat, mwu_p = stats.mannwhitneyu(v1, v2)
    n1, n2 = len(v1), len(v2)
    r = (2 * mwu_stat) / (n1 * n2) - 1

    sig = '*' if mwu_p < 0.05 else ''
    print(f"  {metric:<8} {v1.median():>8.1f}% {v2.median():>8.1f}% "
          f"{stats.trim_mean(v1.values, 0.05):>11.1f}% {stats.trim_mean(v2.values, 0.05):>11.1f}% "
          f"{r:>+7.4f} {mwu_p:>8.4f} {sig:>4}")

    bin_results.append({
        'metric': metric, 'r': r, 'p': mwu_p,
        'med_1': v1.median(), 'med_2p': v2.median(),
        'tm_1': stats.trim_mean(v1.values, 0.05),
        'tm_2p': stats.trim_mean(v2.values, 0.05),
    })

print("\n  Note: Positive r means 1 CN group grew more than 2+ CN group.")
print("  (If more CNs help suppress, we'd expect NEGATIVE r.)")

# --- Part C: Confound check ---
print("\n\n--- PART C: Confound Check — Baseline Engagement by Dosage ---\n")
print("  (Did tweets with more CNs have different baseline engagement?)\n")
print(f"  {'Num_of_CNs':>10} {'n':>5} {'Med Views₀':>12} {'Med Likes₀':>12} {'Med Shares₀':>12}")
print(f"  {'-'*55}")
for n in sorted(trt['Num_of_CNs'].unique()):
    sub = trt[trt['Num_of_CNs'] == n]
    print(f"  {n:>10} {len(sub):>5} {sub['Views_Day0'].median():>11.0f} "
          f"{sub['Likes_Day0'].median():>11.0f} {sub['Shares_Day0'].median():>11.0f}")

stat, p = stats.kruskal(*[trt[trt['Num_of_CNs'] == n]['Views_Day0'].dropna().values
                           for n in range(1, 6) if len(trt[trt['Num_of_CNs'] == n]) >= 3])
print(f"\n  KW test (baseline Views across dosage levels): stat={stat:.2f}, p={p:.4f}")
if p < 0.05:
    print("  ⚠️ Baseline engagement DIFFERS by dosage — more CNs were deployed on different tweets")
    print("  → Dose-response comparison is CONFOUNDED. Interpret with caution.")
else:
    print("  ✅ No significant baseline difference — dose-response comparison is cleaner.")

In [ ]:
# --------------------------------------------------
# 4.5: BOX PLOTS — GROWTH BY CN DOSAGE
# --------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    col = f'{metric}_Growth_13d'

    # Cap for visualization
    cap = trt[col].quantile(0.95)
    floor = max(trt[col].quantile(0.05), -100)

    data_by_dose = []
    labels = []
    for n in sorted(trt['Num_of_CNs'].unique()):
        vals = trt[trt['Num_of_CNs'] == n][col].dropna().clip(floor, cap)
        if len(vals) >= 3:
            data_by_dose.append(vals.values)
            labels.append(f"{n} CN\n(n={len(vals)})")

    # Also add binned
    data_by_dose.append(trt[trt['CN_Bin'] == '1 CN'][col].dropna().clip(floor, cap).values)
    labels.append(f"1 CN\n(n={len(cn1)})")
    data_by_dose.append(trt[trt['CN_Bin'] == '2+ CNs'][col].dropna().clip(floor, cap).values)
    labels.append(f"2+ CNs\n(n={len(cn2p)})")

    bp = ax.boxplot(data_by_dose, tick_labels=labels, patch_artist=True,
                    widths=0.6, showfliers=False)

    # Color: individual doses in blue, binned in green
    n_individual = len(trt['Num_of_CNs'].unique())
    for i, box in enumerate(bp['boxes']):
        if i < n_individual:
            box.set_facecolor('#AED6F1')
            box.set_edgecolor('#2980b9')
        else:
            box.set_facecolor('#A9DFBF')
            box.set_edgecolor('#27ae60')

    # Add vertical separator before binned
    ax.axvline(n_individual + 0.5, color='grey', linestyle=':', alpha=0.5)

    ax.set_ylabel(f'{metric} Growth % (13d)')
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
    ax.grid(True, axis='y', alpha=0.2)

fig.suptitle('4.5: 13-day Growth by CN Dosage (Treatment only)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_5_dosage.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6: KPI — What Was the CN Optimized For?

### Motivation

Each CN was optimized for one of three KPIs:
- **Persuasiveness** (n=225) — designed to be convincing and change minds
- **Shareability** (n=199) — designed to be shared/amplified
- **Emotional Engagement** (n=168) — designed to provoke emotional response

This is a key design question: **does the CN's optimization target affect how well it suppresses the harmful tweet's engagement?** A CN optimized for Persuasiveness might change how people think about the content (reducing Likes), while one optimized for Shareability might generate competing content (boosting Views but not necessarily suppressing the original).

### Design

Treatment-only analysis — we compare growth across the 3 KPI categories within the Treatment group. All three categories are well-powered (n ≥ 168).

In [ ]:
# --------------------------------------------------
# 4.6: KPI — CN OPTIMIZATION TARGET
# --------------------------------------------------

METRICS = ['Views', 'Likes', 'Shares']
KPI_ORDER = ['Persuasiveness', 'Shareability', 'Emotional Engagement']
trt = complete_df[complete_df['Group'] == 'Treatment']

print("=" * 90)
print("4.6: KPI — CN OPTIMIZATION TARGET (TREATMENT-ONLY ANALYSIS)")
print("=" * 90)

# --- Kruskal-Wallis across 3 KPI categories ---
print("\n--- Kruskal-Wallis Test: Does growth differ by KPI? ---\n")
print(f"  {'Metric':<8} {'KW stat':>8} {'p':>8} {'Sig':>4}")
print(f"  {'-'*30}")

for metric in METRICS:
    col = f'{metric}_Growth_13d'
    groups = [trt[trt['KPI'] == kpi][col].dropna().values for kpi in KPI_ORDER]
    kw_stat, kw_p = stats.kruskal(*groups)
    sig = '*' if kw_p < 0.05 else ''
    print(f"  {metric:<8} {kw_stat:>8.2f} {kw_p:>8.4f} {sig:>4}")

# --- Detailed comparison ---
print("\n\n--- Detailed Comparison by KPI ---\n")

kpi_results = []
for metric in METRICS:
    col = f'{metric}_Growth_13d'
    print(f"  {metric}:")
    print(f"    {'KPI':<24} {'n':>5} {'Median':>9} {'TrMean':>9} {'Mean':>10}")
    print(f"    {'-'*60}")

    for kpi in KPI_ORDER:
        vals = trt[trt['KPI'] == kpi][col].dropna()
        print(f"    {kpi:<24} {len(vals):>5} {vals.median():>8.1f}% "
              f"{stats.trim_mean(vals.values, 0.05):>8.1f}% {vals.mean():>9.1f}%")

        kpi_results.append({
            'metric': metric, 'kpi': kpi, 'n': len(vals),
            'median': vals.median(), 'trimmed_mean': stats.trim_mean(vals.values, 0.05),
            'mean': vals.mean(),
        })
    print()

# --- Pairwise comparisons ---
print("\n--- Pairwise MWU Comparisons ---\n")
from itertools import combinations

for metric in METRICS:
    col = f'{metric}_Growth_13d'
    print(f"  {metric}:")
    print(f"    {'Comparison':<40} {'r':>7} {'p':>8} {'Sig':>4}")
    print(f"    {'-'*60}")

    for kpi_a, kpi_b in combinations(KPI_ORDER, 2):
        va = trt[trt['KPI'] == kpi_a][col].dropna()
        vb = trt[trt['KPI'] == kpi_b][col].dropna()
        stat_mwu, p_mwu = stats.mannwhitneyu(va, vb)
        r_mwu = (2 * stat_mwu) / (len(va) * len(vb)) - 1
        sig = '*' if p_mwu < 0.05 else ''
        print(f"    {kpi_a} vs {kpi_b:<20} {r_mwu:>+7.4f} {p_mwu:>8.4f} {sig:>4}")
    print()

# --- Confound check ---
print("\n--- Confound Check: Baseline Engagement by KPI ---\n")
print(f"  {'KPI':<24} {'n':>5} {'Med Views₀':>12} {'Med Likes₀':>12} {'Med Shares₀':>12}")
print(f"  {'-'*70}")
for kpi in KPI_ORDER:
    sub = trt[trt['KPI'] == kpi]
    print(f"  {kpi:<24} {len(sub):>5} {sub['Views_Day0'].median():>11.0f} "
          f"{sub['Likes_Day0'].median():>11.0f} {sub['Shares_Day0'].median():>11.0f}")

stat, p = stats.kruskal(*[trt[trt['KPI'] == kpi]['Views_Day0'].dropna().values for kpi in KPI_ORDER])
print(f"\n  KW test (baseline Views across KPI): stat={stat:.2f}, p={p:.4f}")
if p < 0.05:
    print("  ⚠️ Baseline engagement DIFFERS by KPI — KPI assignment was not independent of tweet type")
else:
    print("  ✅ No significant baseline difference — KPI comparison is cleaner.")

# Also check KPI × narrative cluster and KPI × Type
print("\n\n--- KPI × Narrative Cluster cross-tab ---\n")
ct = pd.crosstab(trt['Cluster_Label'], trt['KPI'])
print(ct.to_string())
chi2, p_chi, _, _ = stats.chi2_contingency(ct)
print(f"\n  Chi-squared test: χ²={chi2:.2f}, p={p_chi:.4f}")
if p_chi < 0.05:
    print("  ⚠️ KPI assignment is NOT independent of narrative cluster")
else:
    print("  ✅ KPI assignment is independent of narrative cluster")

print("\n--- KPI × Tweet Type cross-tab ---\n")
ct2 = pd.crosstab(trt['Type'], trt['KPI'])
print(ct2.to_string())
chi2_2, p_chi2, _, _ = stats.chi2_contingency(ct2)
print(f"\n  Chi-squared test: χ²={chi2_2:.2f}, p={p_chi2:.4f}")

In [ ]:
# --------------------------------------------------
# 4.6: BOX PLOTS — GROWTH BY KPI
# --------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

kpi_colors = {'Persuasiveness': '#2ecc71', 'Shareability': '#3498db', 'Emotional Engagement': '#e74c3c'}

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    col = f'{metric}_Growth_13d'

    cap = trt[col].quantile(0.95)
    floor = max(trt[col].quantile(0.05), -100)

    data = [trt[trt['KPI'] == kpi][col].dropna().clip(floor, cap).values for kpi in KPI_ORDER]
    labels = [f"{kpi}\n(n={len(trt[trt['KPI'] == kpi])})" for kpi in KPI_ORDER]

    bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.6, showfliers=False)

    for i, (box, kpi) in enumerate(zip(bp['boxes'], KPI_ORDER)):
        box.set_facecolor(kpi_colors[kpi])
        box.set_alpha(0.4)
        box.set_edgecolor(kpi_colors[kpi])

    # Overlay medians as text
    for i, kpi in enumerate(KPI_ORDER):
        vals = trt[trt['KPI'] == kpi][col].dropna()
        med = vals.median()
        ax.text(i + 1, med, f'{med:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_ylabel(f'{metric} Growth % (13d)')
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
    ax.grid(True, axis='y', alpha=0.2)

fig.suptitle('4.6: 13-day Growth by CN Optimization Target (Treatment only)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_6_kpi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --------------------------------------------------
# 4.5–4.6 COMBINED TAKEAWAYS
# --------------------------------------------------

print("=" * 75)
print("4.5–4.6 TAKEAWAYS: Treatment-Only Moderators")
print("=" * 75)

# 4.5 Summary
print("\n4.5: CN Dosage (Num_of_CNs)")
print(f"{'─' * 75}")
print("  Distribution: 89% received 1 CN, 11% received 2-5 CNs")

for res in bin_results:
    direction = "more suppression with 2+" if res['r'] < 0 else "less suppression with 2+"
    print(f"  {res['metric']}: 1CN median={res['med_1']:.0f}%, 2+ median={res['med_2p']:.0f}%, "
          f"r={res['r']:+.3f}, p={res['p']:.3f} → {direction}")

# 4.6 Summary
print("\n\n4.6: KPI (CN Optimization Target)")
print(f"{'─' * 75}")

kpi_df = pd.DataFrame(kpi_results)
for metric in METRICS:
    sub = kpi_df[kpi_df['metric'] == metric]
    best = sub.loc[sub['median'].idxmin()] if metric != 'Views' else sub.loc[sub['median'].idxmax()]

    meds = ', '.join([f"{row['kpi']}: {row['median']:.0f}%" for _, row in sub.iterrows()])
    print(f"  {metric}: {meds}")

    if metric == 'Views':
        print(f"    → Most Views growth: {best['kpi']} (median={best['median']:.0f}%)")
    else:
        print(f"    → Least growth (most suppressive): {best['kpi']} (median={best['median']:.0f}%)")

print("\n  Overall: Which KPI produces the most favorable outcomes?")
print("  (For Views: more growth = more anti-suppression → less desirable)")
print("  (For Likes/Shares: less growth = more suppression → more desirable)")

## 4.7: Unified Feature Importance — XGBoost + SHAP

### Motivation

Sections 4.2–4.6 tested each moderator **one at a time** without controlling for other factors. This creates confounds (e.g., dosage correlated with baseline engagement) and misses **interaction effects** (e.g., CNs might suppress Likes only for certain narrative × type combinations).

XGBoost + SHAP solves both problems:
1. **Simultaneous control:** Each feature's SHAP value reflects its contribution *after accounting for all other features*
2. **Nonlinearity:** Tree splits capture threshold effects (e.g., Detection_Lag matters below 30 min but not after)
3. **Interactions:** SHAP interaction values reveal which feature combinations amplify or dampen the treatment effect — our best chance of finding suppression signals hidden in subgroups

### Two-Model Design

| | Model A (Full Sample) | Model B (Treatment Only) |
|---|---|---|
| **n** | ~1,210 | ~592 |
| **Target** | log(1 + 13d growth) per metric | log(1 + 13d growth) per metric |
| **Key question** | What moderates the C vs T difference? | What makes a CN more/less effective? |
| **Features** | Treatment, Type, Narrative_Cluster, Detection_Lag, log baselines | Num_of_CNs, KPI, Type, Narrative_Cluster, Detection_Lag, log baselines |
| **Key output** | SHAP dependence: Treatment × moderators | Feature importance for CN design |

### Log-Transformed Target

Raw percentage growth has extreme outliers (some tweets grew 5,000%+), causing XGBoost to overfit to a handful of extreme cases and producing negative CV R². We apply `log(1 + growth%)` to compress the tail while preserving ordinality. This is standard practice for multiplicative growth metrics.

### Important Caveat

This analysis is **exploratory and associational**, not causal. The randomization supports the main C vs T comparison (Section 3), but moderator effects are observational. We frame these findings as hypothesis-generating for future targeted experiments.

In [ ]:
# --------------------------------------------------
# 4.7.1: DATA PREPARATION FOR XGBOOST
# --------------------------------------------------

import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# --- Feature engineering ---
model_df = complete_df.copy()

# Binary encoding
model_df['Treatment'] = (model_df['Group'] == 'Treatment').astype(int)
model_df['Is_Post'] = (model_df['Type'] == 'post').astype(int)

# Log-transform baselines (add 1 to handle zeros)
for metric in ['Views', 'Likes', 'Shares']:
    model_df[f'log_Baseline_{metric}'] = np.log1p(model_df[f'{metric}_Day0'])

# Log-transform TARGETS to tame outliers
# Growth can be negative (e.g., -50%), so we shift: log(1 + growth/100) won't work for large negatives
# Instead use: sign(x) * log(1 + |x|) — preserves sign, compresses magnitude
for metric in ['Views', 'Likes', 'Shares']:
    raw = model_df[f'{metric}_Growth_13d']
    model_df[f'log_{metric}_Growth_13d'] = np.sign(raw) * np.log1p(np.abs(raw))

# Narrative cluster already created
# KPI encoding (for Model B)
kpi_map = {'Persuasiveness': 0, 'Shareability': 1, 'Emotional Engagement': 2}
model_df['KPI_encoded'] = model_df['KPI'].map(kpi_map)

# --- Define feature sets ---
FEATURES_A = ['Treatment', 'Is_Post', 'Narrative_Cluster', 'Detection_Lag_Minutes',
              'log_Baseline_Views', 'log_Baseline_Likes', 'log_Baseline_Shares']

FEATURES_B = ['Num_of_CNs', 'KPI_encoded', 'Is_Post', 'Narrative_Cluster',
              'Detection_Lag_Minutes', 'log_Baseline_Views', 'log_Baseline_Likes',
              'log_Baseline_Shares']

# Human-readable names for plots
FEATURE_NAMES_A = {
    'Treatment': 'Treatment (0=C, 1=T)',
    'Is_Post': 'Tweet Type (1=Post)',
    'Narrative_Cluster': 'Narrative Cluster',
    'Detection_Lag_Minutes': 'Detection Lag (min)',
    'log_Baseline_Views': 'log(Baseline Views)',
    'log_Baseline_Likes': 'log(Baseline Likes)',
    'log_Baseline_Shares': 'log(Baseline Shares)',
}

FEATURE_NAMES_B = {
    'Num_of_CNs': 'Number of CNs',
    'KPI_encoded': 'KPI Strategy',
    'Is_Post': 'Tweet Type (1=Post)',
    'Narrative_Cluster': 'Narrative Cluster',
    'Detection_Lag_Minutes': 'Detection Lag (min)',
    'log_Baseline_Views': 'log(Baseline Views)',
    'log_Baseline_Likes': 'log(Baseline Likes)',
    'log_Baseline_Shares': 'log(Baseline Shares)',
}

METRICS = ['Views', 'Likes', 'Shares']
TARGETS = {m: f'log_{m}_Growth_13d' for m in METRICS}

# --- Verify data + show target distribution ---
print("=== MODEL DATA SUMMARY ===\n")

# Show target transformation effect
print("--- Target Distribution (raw vs log-transformed) ---\n")
print(f"  {'Metric':<8} {'Raw Mean':>10} {'Raw Std':>10} {'Raw Max':>10} "
      f"{'Log Mean':>10} {'Log Std':>10} {'Log Max':>10}")
print(f"  {'-'*70}")
for m in METRICS:
    raw = complete_df[f'{m}_Growth_13d'].dropna()
    log = model_df[f'log_{m}_Growth_13d'].dropna()
    print(f"  {m:<8} {raw.mean():>9.0f}% {raw.std():>9.0f}% {raw.max():>9.0f}% "
          f"{log.mean():>9.2f} {log.std():>9.2f} {log.max():>9.2f}")

# Full sample
full_df = model_df[FEATURES_A + list(TARGETS.values())].dropna()
print(f"\nModel A (full sample):     n = {len(full_df)}")

# Treatment only
trt_model = model_df[model_df['Treatment'] == 1]
trt_df = trt_model[FEATURES_B + list(TARGETS.values())].dropna()
print(f"Model B (treatment only):  n = {len(trt_df)}")

print(f"\nTargets (log-transformed): {list(TARGETS.values())}")

### 4.7.2: Model A — Full Sample (What Moderates the Treatment Effect?)

In [ ]:
# --------------------------------------------------
# 4.7.2: MODEL A — TRAIN XGBOOST + COMPUTE SHAP
# --------------------------------------------------

import xgboost as xgb
from sklearn.model_selection import cross_val_score
import shap

model_a_results = {}

print("=" * 80)
print("MODEL A: FULL SAMPLE — XGBoost + SHAP (log-transformed target)")
print("=" * 80)

for metric in METRICS:
    target_col = TARGETS[metric]

    df = model_df[FEATURES_A + [target_col]].dropna()
    X = df[FEATURES_A].values
    y = df[target_col].values

    print(f"\n{'─' * 80}")
    print(f"  {metric} (n={len(df)}, target: {target_col})")
    print(f"{'─' * 80}")

    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )

    cv_scores = cross_val_score(xgb_model, X, y, cv=5, scoring='r2')
    print(f"  5-fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Per-fold: {[f'{s:.4f}' for s in cv_scores]}")

    if cv_scores.mean() < 0.01:
        print(f"  ⚠️ R² near zero — model explains negligible variance. Interpret SHAP with caution.")
    elif cv_scores.mean() < 0.10:
        print(f"  ⚠️ R² low (<10%) — limited explanatory power. Interpret SHAP with caution.")
    else:
        print(f"  ✅ Model explains meaningful variance ({cv_scores.mean()*100:.1f}%).")

    # Fit on full data for SHAP
    xgb_model.fit(X, y)

    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X)

    print(f"  Computing SHAP interaction values...")
    shap_interaction = explainer.shap_interaction_values(X)

    model_a_results[metric] = {
        'model': xgb_model,
        'X': X,
        'y': y,
        'feature_names': FEATURES_A,
        'shap_values': shap_values,
        'shap_interaction': shap_interaction,
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
        'n': len(df),
    }

    # Feature importance (gain)
    importance = xgb_model.feature_importances_
    print(f"\n  Feature importance (gain):")
    for fname, imp in sorted(zip(FEATURES_A, importance), key=lambda x: -x[1]):
        bar = '█' * int(imp * 50)
        print(f"    {FEATURE_NAMES_A[fname]:<28} {imp:.4f} {bar}")

    # Mean |SHAP|
    mean_shap = np.abs(shap_values).mean(axis=0)
    print(f"\n  Mean |SHAP| values:")
    for fname, ms in sorted(zip(FEATURES_A, mean_shap), key=lambda x: -x[1]):
        bar = '█' * int(ms / max(mean_shap) * 30)
        print(f"    {FEATURE_NAMES_A[fname]:<28} {ms:.4f} {bar}")

    # Treatment × Moderator interactions
    trt_idx = FEATURES_A.index('Treatment')
    print(f"\n  SHAP interaction: Treatment × Moderator (mean |interaction|):")
    interactions = []
    for j, fname in enumerate(FEATURES_A):
        if fname == 'Treatment':
            continue
        mean_int = np.mean(np.abs(shap_interaction[:, trt_idx, j]))
        interactions.append((fname, mean_int))
    interactions.sort(key=lambda x: -x[1])
    for fname, mean_int in interactions:
        bar = '█' * int(mean_int / interactions[0][1] * 20)
        print(f"    Treatment × {FEATURE_NAMES_A[fname]:<24} {mean_int:.4f} {bar}")

print("\n✅ Model A complete for all metrics.")

### 4.7.3: Model A — SHAP Visualizations

In [ ]:
# --------------------------------------------------
# 4.7.3a: SHAP BEESWARM PLOTS — MODEL A
# --------------------------------------------------

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    res = model_a_results[metric]

    plt.sca(ax)
    shap.summary_plot(
        res['shap_values'],
        res['X'],
        feature_names=[FEATURE_NAMES_A[f] for f in FEATURES_A],
        show=False,
        plot_size=None,
        max_display=len(FEATURES_A),
    )
    ax.set_title(f"{metric}\n(CV R² = {res['cv_r2_mean']:.3f})", fontweight='bold', fontsize=12)
    ax.set_xlabel('SHAP value (log-growth scale)')

fig.suptitle('Model A: SHAP Beeswarm — What Drives log(13-day Growth)? (Full Sample)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_7_shap_beeswarm_A.png', dpi=150, bbox_inches='tight')
plt.show()
# --------------------------------------------------
# 4.7.3b: SHAP DEPENDENCE PLOTS — Treatment × Moderators
# --------------------------------------------------

from statsmodels.nonparametric.smoothers_lowess import lowess

MODERATORS = ['Is_Post', 'Narrative_Cluster', 'Detection_Lag_Minutes',
              'log_Baseline_Views', 'log_Baseline_Likes', 'log_Baseline_Shares']

fig, axes = plt.subplots(len(METRICS), len(MODERATORS), figsize=(24, 12))

trt_idx = FEATURES_A.index('Treatment')

for row, metric in enumerate(METRICS):
    res = model_a_results[metric]

    for col_idx, mod in enumerate(MODERATORS):
        ax = axes[row, col_idx]
        mod_idx = FEATURES_A.index(mod)

        trt_shap = res['shap_values'][:, trt_idx]
        mod_values = res['X'][:, mod_idx]

        scatter = ax.scatter(mod_values, trt_shap, c=mod_values, cmap='coolwarm',
                           alpha=0.3, s=12, rasterized=True)

        # LOWESS trend
        valid = ~np.isnan(mod_values) & ~np.isnan(trt_shap)
        if valid.sum() > 50 and len(np.unique(mod_values[valid])) > 3:
            smoothed = lowess(trt_shap[valid], mod_values[valid], frac=0.5, return_sorted=True)
            ax.plot(smoothed[:, 0], smoothed[:, 1], 'k-', linewidth=2.5)

        ax.axhline(0, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)

        if col_idx == 0:
            ax.set_ylabel(f'{metric}\nTreatment SHAP', fontsize=10)
        if row == 0:
            ax.set_title(FEATURE_NAMES_A[mod], fontsize=9, fontweight='bold')
        if row == len(METRICS) - 1:
            ax.set_xlabel(FEATURE_NAMES_A[mod], fontsize=8)

fig.suptitle('Model A: How Each Moderator Changes the Treatment Effect (SHAP Dependence)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_7_shap_dependence_A.png', dpi=150, bbox_inches='tight')
plt.show()

print("Reading these plots:")
print("  X-axis = moderator value")
print("  Y-axis = SHAP value for Treatment feature (log-growth scale)")
print("  If Y > 0: Treatment pushes log-growth UP (anti-suppression)")
print("  If Y < 0: Treatment pushes log-growth DOWN (suppression)")
print("  A SLOPE means the moderator CHANGES the treatment effect")
print("  A FLAT line means the moderator does NOT interact with treatment")
# --------------------------------------------------
# 4.7.3c: SHAP INTERACTION HEATMAP — MODEL A
# --------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    res = model_a_results[metric]

    interaction_matrix = np.abs(res['shap_interaction']).mean(axis=0)
    labels = [FEATURE_NAMES_A[f] for f in FEATURES_A]

    im = ax.imshow(interaction_matrix, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(FEATURES_A)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(FEATURES_A)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_title(f"{metric}\n(CV R² = {res['cv_r2_mean']:.3f})", fontweight='bold', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.8, label='Mean |SHAP interaction|')

    for i in range(len(FEATURES_A)):
        for j in range(len(FEATURES_A)):
            val = interaction_matrix[i, j]
            color = 'white' if val > interaction_matrix.max() * 0.6 else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=6, color=color)

fig.suptitle('Model A: SHAP Interaction Matrix (log-growth target)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_7_shap_interaction_A.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.7.4: Model B — Treatment Only (What Makes a CN More Effective?)

In [ ]:
# --------------------------------------------------
# 4.7.4: MODEL B — TREATMENT ONLY
# --------------------------------------------------

model_b_results = {}

print("=" * 80)
print("MODEL B: TREATMENT ONLY — XGBoost + SHAP (log-transformed target)")
print("=" * 80)

trt_model_df = model_df[model_df['Treatment'] == 1].copy()

for metric in METRICS:
    target_col = TARGETS[metric]

    df = trt_model_df[FEATURES_B + [target_col]].dropna()
    X = df[FEATURES_B].values
    y = df[target_col].values

    print(f"\n{'─' * 80}")
    print(f"  {metric} (n={len(df)})")
    print(f"{'─' * 80}")

    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )

    cv_scores = cross_val_score(xgb_model, X, y, cv=5, scoring='r2')
    print(f"  5-fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Per-fold: {[f'{s:.4f}' for s in cv_scores]}")

    if cv_scores.mean() < 0.01:
        print(f"  ⚠️ R² near zero — model explains negligible variance.")
    elif cv_scores.mean() < 0.10:
        print(f"  ⚠️ R² low (<10%) — limited explanatory power.")
    else:
        print(f"  ✅ Model explains meaningful variance ({cv_scores.mean()*100:.1f}%).")

    xgb_model.fit(X, y)

    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X)

    model_b_results[metric] = {
        'model': xgb_model,
        'X': X,
        'y': y,
        'feature_names': FEATURES_B,
        'shap_values': shap_values,
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
        'n': len(df),
    }

    importance = xgb_model.feature_importances_
    print(f"\n  Feature importance (gain):")
    for fname, imp in sorted(zip(FEATURES_B, importance), key=lambda x: -x[1]):
        bar = '█' * int(imp * 50)
        print(f"    {FEATURE_NAMES_B[fname]:<28} {imp:.4f} {bar}")

    mean_shap = np.abs(shap_values).mean(axis=0)
    print(f"\n  Mean |SHAP| values:")
    for fname, ms in sorted(zip(FEATURES_B, mean_shap), key=lambda x: -x[1]):
        bar = '█' * int(ms / max(mean_shap) * 30)
        print(f"    {FEATURE_NAMES_B[fname]:<28} {ms:.4f} {bar}")

print("\n✅ Model B complete for all metrics.")
# --------------------------------------------------
# 4.7.4b: SHAP BEESWARM — MODEL B
# --------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, metric in enumerate(METRICS):
    ax = axes[idx]
    res = model_b_results[metric]

    plt.sca(ax)
    shap.summary_plot(
        res['shap_values'],
        res['X'],
        feature_names=[FEATURE_NAMES_B[f] for f in FEATURES_B],
        show=False,
        plot_size=None,
        max_display=len(FEATURES_B),
    )
    ax.set_title(f"{metric}\n(CV R² = {res['cv_r2_mean']:.3f})", fontweight='bold', fontsize=12)
    ax.set_xlabel('SHAP value (log-growth scale)')

fig.suptitle('Model B: SHAP Beeswarm — What Predicts CN Effectiveness? (Treatment Only)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_7_shap_beeswarm_B.png', dpi=150, bbox_inches='tight')
plt.show()
# --------------------------------------------------
# 4.7.4c: SHAP DEPENDENCE — KEY FEATURES IN MODEL B
# --------------------------------------------------

KEY_FEATURES_B = ['Num_of_CNs', 'KPI_encoded', 'Detection_Lag_Minutes', 'log_Baseline_Views']

fig, axes = plt.subplots(len(METRICS), len(KEY_FEATURES_B), figsize=(20, 12))

for row, metric in enumerate(METRICS):
    res = model_b_results[metric]

    for col_idx, feat in enumerate(KEY_FEATURES_B):
        ax = axes[row, col_idx]
        feat_idx = FEATURES_B.index(feat)

        feat_vals = res['X'][:, feat_idx]
        feat_shap = res['shap_values'][:, feat_idx]

        ax.scatter(feat_vals, feat_shap, alpha=0.3, s=12, color='#3498db', rasterized=True)

        valid = ~np.isnan(feat_vals) & ~np.isnan(feat_shap)
        if valid.sum() > 50 and len(np.unique(feat_vals[valid])) > 3:
            smoothed = lowess(feat_shap[valid], feat_vals[valid], frac=0.5, return_sorted=True)
            ax.plot(smoothed[:, 0], smoothed[:, 1], 'r-', linewidth=2)

        ax.axhline(0, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)

        if row == len(METRICS) - 1:
            ax.set_xlabel(FEATURE_NAMES_B[feat], fontsize=9)
        if col_idx == 0:
            ax.set_ylabel(f'{metric}\nSHAP value', fontsize=10)
        if row == 0:
            ax.set_title(FEATURE_NAMES_B[feat], fontsize=10, fontweight='bold')

fig.suptitle('Model B: SHAP Dependence — Key Features (Treatment Only)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_4_7_shap_dependence_B.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.7.5: COMPARATIVE SUMMARY — BOTH MODELS

In [ ]:
# --------------------------------------------------
# 4.7.5: COMPARATIVE SUMMARY — BOTH MODELS
# --------------------------------------------------

print("=" * 85)
print("4.7 SUMMARY: XGBoost + SHAP Results (log-transformed target)")
print("=" * 85)

# --- Model A Summary ---
print("\n╔══════════════════════════════════════════════════════════════════════════════════╗")
print("║  MODEL A: Full Sample — What moderates the Treatment effect?                    ║")
print("╚══════════════════════════════════════════════════════════════════════════════════╝")

print(f"\n  {'Metric':<8} {'CV R²':>10} {'Top Feature':>28} {'Trt Rank':>10} {'Trt mean|SHAP|':>16}")
print(f"  {'─'*75}")

for metric in METRICS:
    res = model_a_results[metric]
    mean_shap = np.abs(res['shap_values']).mean(axis=0)
    ranked = sorted(zip(FEATURES_A, mean_shap), key=lambda x: -x[1])
    top_feat = FEATURE_NAMES_A[ranked[0][0]]
    trt_rank = [f for f, _ in ranked].index('Treatment') + 1
    trt_shap_val = mean_shap[FEATURES_A.index('Treatment')]
    print(f"  {metric:<8} {res['cv_r2_mean']:>9.4f} {top_feat:>28} {trt_rank:>6}/7 {trt_shap_val:>14.4f}")

# Treatment × moderator interaction summary
print(f"\n  Strongest Treatment × Moderator interactions (Model A):")
trt_idx = FEATURES_A.index('Treatment')
for metric in METRICS:
    res = model_a_results[metric]
    interactions = []
    for j, fname in enumerate(FEATURES_A):
        if fname == 'Treatment':
            continue
        mean_int = np.mean(np.abs(res['shap_interaction'][:, trt_idx, j]))
        interactions.append((fname, mean_int))
    interactions.sort(key=lambda x: -x[1])
    top3 = interactions[:3]
    top_str = ', '.join([f"{FEATURE_NAMES_A[f]}={v:.4f}" for f, v in top3])
    print(f"    {metric}: {top_str}")

# --- Model B Summary ---
print("\n╔══════════════════════════════════════════════════════════════════════════════════╗")
print("║  MODEL B: Treatment Only — What makes a CN more effective?                      ║")
print("╚══════════════════════════════════════════════════════════════════════════════════╝")

print(f"\n  {'Metric':<8} {'CV R²':>10} {'Top 3 Features (by mean |SHAP|)':>55}")
print(f"  {'─'*75}")

for metric in METRICS:
    res = model_b_results[metric]
    mean_shap = np.abs(res['shap_values']).mean(axis=0)
    ranked = sorted(zip(FEATURES_B, mean_shap), key=lambda x: -x[1])
    top3 = ', '.join([FEATURE_NAMES_B[f] for f, _ in ranked[:3]])
    print(f"  {metric:<8} {res['cv_r2_mean']:>9.4f} {top3:>55}")

# --- Treatment SHAP direction analysis ---
print(f"\n{'═' * 85}")
print("TREATMENT EFFECT DIRECTION ANALYSIS (Model A)")
print(f"{'═' * 85}")

for metric in METRICS:
    res = model_a_results[metric]
    trt_shap_vals = res['shap_values'][:, FEATURES_A.index('Treatment')]

    pct_negative = (trt_shap_vals < 0).mean() * 100
    pct_positive = (trt_shap_vals > 0).mean() * 100
    pct_zero = (trt_shap_vals == 0).mean() * 100
    mean_val = trt_shap_vals.mean()

    print(f"\n  {metric}:")
    print(f"    Mean Treatment SHAP: {mean_val:+.4f}")
    print(f"    Direction: {pct_negative:.1f}% negative (suppression), "
          f"{pct_positive:.1f}% positive (anti-suppression), {pct_zero:.1f}% zero")

    if pct_negative > 40 and pct_positive > 40:
        print(f"    → MIXED: Treatment effect varies across tweets — check dependence plots")
    elif pct_negative > 60:
        print(f"    → PREDOMINANTLY SUPPRESSIVE")
    elif pct_positive > 60:
        print(f"    → PREDOMINANTLY ANTI-SUPPRESSIVE")
    else:
        print(f"    → WEAK/NEGLIGIBLE treatment contribution")

# --- Suppression subgroup search ---
print(f"\n{'═' * 85}")
print("SUBGROUP SEARCH: Where does Treatment push growth DOWN?")
print(f"{'═' * 85}")
print("\n  Checking if any moderator values consistently produce NEGATIVE Treatment SHAP:")

for metric in METRICS:
    res = model_a_results[metric]
    X_df = pd.DataFrame(res['X'], columns=FEATURES_A)
    trt_shap_vals = res['shap_values'][:, FEATURES_A.index('Treatment')]

    print(f"\n  {metric}:")

    # Check by tweet type
    for ttype, tval in [('Comment', 0), ('Post', 1)]:
        mask = X_df['Is_Post'] == tval
        sub_shap = trt_shap_vals[mask]
        pct_neg = (sub_shap < 0).mean() * 100
        print(f"    {ttype}: mean Trt SHAP = {sub_shap.mean():+.4f}, {pct_neg:.0f}% negative")

    # Check by narrative cluster
    for cid in sorted(CLUSTER_LABELS.keys()):
        mask = X_df['Narrative_Cluster'] == cid
        if mask.sum() < 20:
            continue
        sub_shap = trt_shap_vals[mask]
        pct_neg = (sub_shap < 0).mean() * 100
        if pct_neg > 55:  # flag clusters where >55% of Treatment SHAP is negative
            print(f"    Cluster {cid} ({CLUSTER_LABELS[cid]}): mean={sub_shap.mean():+.4f}, "
                  f"{pct_neg:.0f}% negative ← POTENTIAL SUPPRESSION SUBGROUP")
        else:
            print(f"    Cluster {cid} ({CLUSTER_LABELS[cid]}): mean={sub_shap.mean():+.4f}, "
                  f"{pct_neg:.0f}% negative")

    # Check by detection lag tercile
    lag_col = X_df['Detection_Lag_Minutes']
    for label, lo, hi in [('Fast (<42min)', 0, 42), ('Medium (42-109)', 42, 109), ('Slow (>109)', 109, 999)]:
        mask = (lag_col >= lo) & (lag_col < hi)
        if mask.sum() < 20:
            continue
        sub_shap = trt_shap_vals[mask]
        pct_neg = (sub_shap < 0).mean() * 100
        print(f"    {label}: mean={sub_shap.mean():+.4f}, {pct_neg:.0f}% negative")

print()